# Análisis del proceso de entrenamiento — cadena curricular hasta E2.1 (STH-WP y SUB-WP)

Este cuaderno analiza el proceso progresivo de entrenamiento por refuerzo (PPO) de las
arquitecturas **STH-WP** (Single-Task Heading Waypoint) y **SUB-WP** (Sub-Waypoint continuo),
desde las fases iniciales de aproximación hasta la configuración final **E2.1**, incluyendo las
dos ramas experimentales posteriores **E2.2** y **E2.2b**.

El alcance se limita estrictamente a la cadena válida verificada mediante inspección directa de
los scripts de entrenamiento y de las llamadas `PPO.load(...)` que determinan la procedencia real
de cada modelo. No se analizan evaluaciones de generalización (WH02–WH05), evaluaciones R0/R1,
entrenamientos abandonados (E1, E1_pred, E1.2–E1.5), ramas dinámicas antiguas, ni versiones
anteriores a las correcciones r2 (SUB-WP) o v2 (STH-WP).

**Restricciones respetadas en la elaboración de este cuaderno:** no se ha ejecutado Webots, no se
han lanzado nuevos entrenamientos ni inferencias, no se han modificado checkpoints, CSV ni
registros de TensorBoard originales, y no se ha sobrescrito ningún archivo ya existente en el
repositorio.

## 1. Objetivo y alcance

Este cuaderno persigue explicar cómo se entrenaron progresivamente las políticas de navegación,
qué comportamiento aporta cada fase curricular, cómo evolucionan las métricas funcionales
(éxito, colisión, truncamiento) y las métricas internas de optimización de PPO durante el
entrenamiento, y por qué la configuración **E2.1** se seleccionó como modelo final frente a las
ramas experimentales **E2.2** y **E2.2b**.

La cadena analizada es la siguiente, verificada mediante lectura directa de los scripts de
entrenamiento (no inferida de los nombres de archivo):

- **STH-WP**: Fase 1 → Fase 2 → Fase 3v2 → Fase 4v2 → {Fase 5, Fase 6} → E2.1 → {E2.2, E2.2b}
- **SUB-WP**: Fase 1 → Fase 2 → Fase 3 r2 → Fase 4 r2 → Fase 5 r2 → E2.0 → E2.1 → {E2.2, E2.2b}

Como se detalla en la Sección 4, la auditoría reveló que en STH-WP las fases 5 y 6 **no son
secuenciales**: ambas parten directamente de Fase 4v2 y constituyen ramas paralelas con
propósitos distintos (retorno puro y ciclo completo, respectivamente). Este hallazgo se muestra
de forma explícita en el diagrama de cadena y no se ha forzado ninguna relación secuencial no
respaldada por el código.

## 2. Metodología

El análisis combina tres fuentes de información, todas ellas de solo lectura:

1. **Scripts de entrenamiento** (`train_stage*.py`, `experimentos/scripts/sthwp_e2_*.py`,
   `experimentos/scripts/subwp_e2_*.py`), inspeccionados para localizar las llamadas
   `PPO.load(...)` que determinan qué checkpoint alimenta cada fase, así como los parámetros de
   entorno (dimensión de observación, alcance del LIDAR, presencia de peatón) y el presupuesto de
   pasos de entrenamiento (`total_timesteps`).
2. **Registros de TensorBoard** (`tensorboard_logs/`), leídos mediante `EventAccumulator` para
   extraer tanto las métricas funcionales registradas por un *callback* personalizado
   (`stats/tasa_exito_%`, `stats/tasa_colision_%`, `stats/tasa_truncado_%`, resultados por
   objetivo, etc.) como las métricas internas de optimización de PPO (`train/value_loss`,
   `train/policy_gradient_loss`, `train/entropy_loss`, `train/approx_kl`, `train/clip_fraction`,
   `train/explained_variance`, `train/learning_rate`).
3. **CSV de estadísticas finales** (`stats_*.csv`), que contienen una tabla agregada por objetivo
   (28 estanterías) calculada al final de cada fase de entrenamiento. Estos archivos **no son
   series temporales**: constituyen una fotografía final, complementaria a la evolución temporal
   disponible en TensorBoard.

Se distingue explícitamente entre **métricas funcionales** (tasa de éxito, colisión, truncamiento,
longitud de episodio, llegada a estantería) y **métricas internas de optimización** de PPO. La
recompensa (`rollout/ep_rew_mean`) nunca se emplea como sustituto de la tasa de éxito.

Cuando una fase carece de un dato (CSV inexistente, directorio de TensorBoard no localizado,
semilla ausente), el cuaderno lo declara explícitamente como **"no registrada"** o **"no
disponible"** en lugar de inventar o interpolar un valor.

**Nota metodológica sobre el eje de pasos**: se ha comprobado que algunos scripts de SUB-WP
(fases 3 r2, 4 r2 y 5 r2) entrenan con `reset_num_timesteps=False`, por lo que el contador de
pasos registrado en TensorBoard es acumulado desde el inicio de la ejecución completa y no reinicia
en cada fase. Para evitar interpretar esto como presupuestos de pasos distintos a los indicados en
el script, todas las curvas de este cuaderno se representan en un **eje de pasos relativo**,
calculado restando el primer paso registrado en cada directorio de TensorBoard.

In [1]:
# Configuración general, imports y utilidades comunes
import os
import re
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

warnings.filterwarnings("ignore")

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
NB_DIR = os.path.abspath(os.getcwd())
FIG_DIR = os.path.join(NB_DIR, "figuras_entrenamiento_hasta_e2_1")
TAB_DIR = os.path.join(NB_DIR, "resultados_entrenamiento_hasta_e2_1")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

STH_BASE = os.path.join(REPO_ROOT, "simulacion", "controllers", "rl_train_STHWP")
SUB_BASE = os.path.join(REPO_ROOT, "simulacion", "controllers", "rl_train_SUB_WP_continuo")

SEED_COLORS = {42: "#1F77B4", 123: "#E6A700", 524: "#2CA02C"}  # azul, ocre, verde
ARCH_COLORS = {"STH-WP": "#1F77B4", "SUB-WP": "#FF7F0E"}
SEEDS = [42, 123, 524]

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "axes.grid": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "grid.color": "#D9D9D9",
    "grid.alpha": 0.55,
    "grid.linestyle": "--",
    "font.size": 10,
})

print("REPO_ROOT:", REPO_ROOT)
print("STH_BASE existe:", os.path.isdir(STH_BASE))
print("SUB_BASE existe:", os.path.isdir(SUB_BASE))

REPO_ROOT: /Users/adrigarcia/tfm-robots-rl
STH_BASE existe: True
SUB_BASE existe: True


## 3. Auditoría de datos\n\nEsta sección implementa las funciones reutilizables de lectura de TensorBoard y CSV, y registra advertencias explícitas sobre archivos vacíos, pasos duplicados o directorios no identificables.

In [2]:
# --- Funciones reutilizables de TensorBoard ---
WARNINGS_LOG = []

def _warn(msg):
    WARNINGS_LOG.append(msg)

def find_tb_dir(base, name):
    """Localiza un directorio de TensorBoard por nombre exacto bajo tensorboard_logs/."""
    path = os.path.join(base, "tensorboard_logs", name)
    if os.path.isdir(path):
        ev_files = glob.glob(os.path.join(path, "events.out.tfevents.*"))
        if not ev_files:
            _warn(f"Directorio TB sin archivos de eventos: {path}")
            return None, []
        return path, ev_files
    _warn(f"Directorio TB no encontrado: {path}")
    return None, []

def list_scalar_tags(tb_dir):
    """Lista los tags escalares disponibles en un directorio de TensorBoard."""
    if tb_dir is None:
        return []
    try:
        ea = EventAccumulator(tb_dir, size_guidance={"scalars": 0})
        ea.Reload()
        return sorted(ea.Tags().get("scalars", []))
    except Exception as e:
        _warn(f"Error leyendo tags de {tb_dir}: {e}")
        return []

def read_scalar(tb_dir, tag, relative_steps=True):
    """Lee una métrica escalar de un directorio de TensorBoard.

    Devuelve un DataFrame con columnas step, value, wall_time. Si relative_steps=True,
    el step se normaliza restando el primer step registrado en ese directorio (necesario
    porque algunas fases de SUB-WP no reinician el contador global de timesteps).
    Devuelve DataFrame vacío si el tag no existe (se registra advertencia).
    """
    if tb_dir is None:
        return pd.DataFrame(columns=["step", "value", "wall_time"])
    try:
        ea = EventAccumulator(tb_dir, size_guidance={"scalars": 0})
        ea.Reload()
        if tag not in ea.Tags().get("scalars", []):
            return pd.DataFrame(columns=["step", "value", "wall_time"])
        evs = ea.Scalars(tag)
        if not evs:
            _warn(f"Tag '{tag}' vacío en {tb_dir}")
            return pd.DataFrame(columns=["step", "value", "wall_time"])
        df = pd.DataFrame([(e.step, e.value, e.wall_time) for e in evs],
                           columns=["step", "value", "wall_time"])
        n_before = len(df)
        df = df.drop_duplicates(subset="step", keep="last").sort_values("step").reset_index(drop=True)
        if len(df) < n_before:
            _warn(f"Pasos duplicados eliminados en {tb_dir} / {tag}: {n_before - len(df)}")
        # Deteccion de reinicio: el step disminuye en algun punto de la secuencia original
        raw_steps = [e.step for e in evs]
        if any(raw_steps[i] > raw_steps[i+1] for i in range(len(raw_steps)-1)):
            _warn(f"Reinicio de contador de pasos detectado en {tb_dir} / {tag} (no se ha corregido automáticamente)")
        if relative_steps and len(df) > 0:
            df["step"] = df["step"] - df["step"].iloc[0]
        return df
    except Exception as e:
        _warn(f"Error leyendo '{tag}' de {tb_dir}: {e}")
        return pd.DataFrame(columns=["step", "value", "wall_time"])

def max_step_raw(tb_dir, tag="rollout/ep_len_mean"):
    """Devuelve el ultimo step (sin normalizar) registrado para un tag, o None."""
    if tb_dir is None:
        return None
    try:
        ea = EventAccumulator(tb_dir, size_guidance={"scalars": 0})
        ea.Reload()
        tags = ea.Tags().get("scalars", [])
        use_tag = tag if tag in tags else (tags[0] if tags else None)
        if use_tag is None:
            return None
        evs = ea.Scalars(use_tag)
        return evs[-1].step if evs else None
    except Exception:
        return None

def ema(series, alpha=0.85):
    """Media movil exponencial con el factor de suavizado pedido (0.85)."""
    if len(series) == 0:
        return series
    out = np.zeros(len(series))
    out[0] = series.iloc[0] if hasattr(series, "iloc") else series[0]
    for i in range(1, len(series)):
        v = series.iloc[i] if hasattr(series, "iloc") else series[i]
        out[i] = alpha * out[i-1] + (1 - alpha) * v
    return out

def read_stats_csv(path):
    """Lee un CSV de estadisticas agregadas por objetivo. Devuelve None si no existe."""
    if path is None or not os.path.isfile(path):
        return None
    try:
        return pd.read_csv(path)
    except Exception as e:
        _warn(f"Error leyendo CSV {path}: {e}")
        return None

print("Funciones de auditoria y lectura de TensorBoard definidas.")

Funciones de auditoria y lectura de TensorBoard definidas.


In [3]:
# --- Definicion de la cadena de fases (registro verificado por auditoria manual) ---
# Cada entrada describe una fase entrenada. 'seeds' es un dict seed -> info especifica
# (permite manejar asimetrias como la de SUB-WP E2.0 semilla 123).

def phase(arch, name, order, objetivo, tb_template, csv_template, ckpt_origen, ckpt_generado,
          timesteps_previstos, seeds=SEEDS, tb_overrides=None, csv_overrides=None,
          ckpt_origen_overrides=None, notas=""):
    tb_overrides = tb_overrides or {}
    csv_overrides = csv_overrides or {}
    ckpt_origen_overrides = ckpt_origen_overrides or {}
    return dict(arch=arch, name=name, order=order, objetivo=objetivo,
                tb_template=tb_template, csv_template=csv_template,
                ckpt_origen=ckpt_origen, ckpt_generado=ckpt_generado,
                timesteps_previstos=timesteps_previstos, seeds=seeds,
                tb_overrides=tb_overrides, csv_overrides=csv_overrides,
                ckpt_origen_overrides=ckpt_origen_overrides, notas=notas)

STHWP_PHASES = [
    phase("STH-WP", "Fase 1", 1, "Aproximacion inicial a una ubicacion",
          "stage1_run002_s{seed}_1", "stats_run002_s{seed}_stage1.csv",
          "desde cero", "pruebas/run002_s{seed}_stage1_final.zip", 500_000,
          notas="Existe tambien run001 (primer intento de fases 1-3), excluido de la cadena valida porque no continua a 3v2."),
    phase("STH-WP", "Fase 2", 2, "Aproximacion generalizada a las 28 ubicaciones",
          "stage2_run002_s{seed}_0", "stats_run002_s{seed}_stage2.csv",
          "pruebas/run002_s{seed}_stage1_final.zip", "pruebas/run002_s{seed}_stage2_final.zip", 4_000_000),
    phase("STH-WP", "Fase 3v2", 3, "Salida desde la estanteria (version corregida v2)",
          "stage3_i6_run002_s{seed}_0", None,
          "pruebas/run002_s{seed}_stage2_final.zip", "pruebas/run002_s{seed}_stage3v2_final.zip", 4_000_000,
          notas="tb_log_name real es 'stage3_i6_...', no existe carpeta 'stage3v2_*' (trampa de nomenclatura verificada en el script). "
                "No existe train_stage3v2_s524.py en el repositorio: el checkpoint y su inferencia existen pero el script de origen para "
                "esta semilla no ha podido localizarse. No existe CSV de entrenamiento especifico de esta fase."),
    phase("STH-WP", "Fase 4v2", 4, "Combinacion de aproximacion y salida",
          "stage4v2_run002_s{seed}_0", None,
          "pruebas/run002_s{seed}_stage3v2_final.zip", "pruebas/run002_s{seed}_stage4v2_final.zip", 4_000_000,
          notas="No existe CSV de entrenamiento para esta fase, solo TensorBoard."),
    phase("STH-WP", "Fase 5", 5, "Retorno a la base (rama de retorno puro)",
          "stage5_run002_s{seed}_0", "stats_run002_s{seed}_stage5.csv",
          "pruebas/run002_s{seed}_stage4v2_final.zip", "pruebas/run002_s{seed}_stage5_final.zip", 2_000_000,
          notas="Parte directamente de Fase 4v2. NO continua hacia Fase 6: son ramas paralelas independientes, verificado en train_stage6_s*.py."),
    phase("STH-WP", "Fase 6", 6, "Ciclo completo (aproximacion + salida + retorno, entrenamiento end-to-end)",
          "stage6_run003_s{seed}_0", "stats_run003_s{seed}_stage6.csv",
          "pruebas/run002_s{seed}_stage4v2_final.zip", "pruebas/run003_s{seed}_stage6_final.zip", 1_000_000,
          notas="CONFIRMADO: Fase 6 carga el checkpoint de Fase 4v2, NO el de Fase 5. Fases 5 y 6 son ramas paralelas "
                "desde un mismo origen comun, no una cadena secuencial 4v2->5->6. Existe un intento previo descartado "
                "'run002_s{42,524}_stage6_final' (docstring desactualizado, ausente para s123) que NO se usa en este cuaderno."),
    phase("STH-WP", "E2.1", 7, "Comportamiento con peaton mediante observaciones LIDAR realistas",
          "sthwp_e2_1_s{seed}_1", "stats_sthwp_e2_1_s{seed}_stage6.csv",
          "pruebas/run003_s{seed}_stage6_final.zip", "pruebas/sthwp_e2_1_s{seed}_final.zip", 6_000_000,
          notas="Observacion de 40 dimensiones (36 LIDAR + 4 estado), LIDAR ampliado a 5.0 m, sin posicion/velocidad de peaton "
                "en la observacion de la politica, objetivos dificiles (goal_14..goal_28) ponderados 3x. "
                "MATIZ METODOLOGICO: la recompensa usa internamente la posicion del peaton via supervisor aunque la politica no la observe."),
    phase("STH-WP", "E2.2", 8, "Replanning dinamico basado en LIDAR (umbral 1.5-4.5 m)",
          "sthwp_e2_2_s{seed}_1", "stats_sthwp_e2_2_s{seed}_stage6.csv",
          "pruebas/sthwp_e2_1_s{seed}_final.zip", "pruebas/sthwp_e2_2_s{seed}_final.zip", 6_000_000,
          notas="Parte de E2.1. Filtro de deteccion dinamica LIDAR en [1.5, 4.5] m. No se presupone mejora sobre E2.1."),
    phase("STH-WP", "E2.2b", 9, "Replanning dinamico basado en LIDAR (umbral corregido 2.5-4.5 m)",
          "sthwp_e2_2b_s{seed}_1", "stats_sthwp_e2_2b_s{seed}_stage6.csv",
          "pruebas/sthwp_e2_1_s{seed}_final.zip", "pruebas/sthwp_e2_2b_s{seed}_final.zip", 6_000_000,
          notas="Parte DIRECTAMENTE de E2.1 (no de E2.2). Umbral minimo de deteccion elevado de 1.5 a 2.5 m para "
                "reducir falsos positivos en pasillos estrechos (~2 m de ancho). No se presupone superioridad sobre E2.1."),
]

SUBWP_PHASES = [
    phase("SUB-WP", "Fase 1", 1, "Aproximacion inicial a una ubicacion",
          "stage1_subwp_s{seed}_1", "stats_subwp_s{seed}_stage1.csv",
          "desde cero", "pruebas/subwp_s{seed}_stage1_final.zip", 500_000,
          notas="Para s524 existe tambien 'stage1_subwp_s524_2' (ejecucion independiente duplicada, no una continuacion)."),
    phase("SUB-WP", "Fase 2", 2, "Aproximacion generalizada a las 28 ubicaciones",
          "stage2_subwp_s{seed}_0", "stats_subwp_s{seed}_stage2.csv",
          "pruebas/subwp_s{seed}_stage1_final.zip", "pruebas/subwp_s{seed}_stage2_final.zip", 4_000_000),
    phase("SUB-WP", "Fase 3 r2", 3, "Salida desde la estanteria (correccion r2 de la zona de descarga)",
          "stage3_r2_subwp_s{seed}_wp75_noped_0", "stats_subwp_s{seed}_r2_stage3.csv",
          "pruebas/subwp_s{seed}_stage2_final.zip", "pruebas/subwp_s{seed}_wp75_r2_stage3_final.zip", 4_000_000,
          tb_overrides={42: "stage3_r2_subwp_s42_wp75_noped_0"},
          notas="La correccion r2 resuelve un error en la zona de descarga (dropoff) que invalidaba las fases de "
                "salida y retorno anteriores (documentado explicitamente en el docstring del script). Se excluyen "
                "las versiones sin r2. Para s42 existe tambien 'stage3_r2_subwp_s42_wp75_0' (sin sufijo noped), un "
                "intento mas corto (~4.6M pasos acumulados) que no llega al mismo punto que la version 'noped' "
                "(~8.5M acumulados, coherente con las otras semillas); se usa la version 'noped' como canonica."),
    phase("SUB-WP", "Fase 4 r2", 4, "Ciclo aproximacion + salida (70/30)",
          "stage4_r2_subwp_s{seed}_wp75_noped_0", "stats_subwp_s{seed}_r2_stage4.csv",
          "pruebas/subwp_s{seed}_wp75_r2_stage3_final.zip", "pruebas/subwp_s{seed}_wp75_r2_stage4_final.zip", 4_000_000),
    phase("SUB-WP", "Fase 5 r2", 5, "Retorno a la base (correccion r2)",
          "stage5_r2_subwp_s{seed}_wp75_noped_0", "stats_subwp_s{seed}_r2_stage5.csv",
          "pruebas/subwp_s{seed}_wp75_r2_stage4_final.zip", "pruebas/subwp_s{seed}_wp75_r2_stage5_final.zip", 2_000_000),
    phase("SUB-WP", "E2.0", 6, "Ciclo completo sin observacion de peaton (equivalente a Fase 6 de STH-WP)",
          "subwp_e2_0_s{seed}_1", "stats_subwp_e2_0_s{seed}_stage6.csv",
          "pruebas/subwp_s{seed}_wp75_r2_stage5_final.zip", "pruebas/subwp_e2_0_s{seed}_final.zip", 2_000_000,
          tb_overrides={123: "subwp_e2_0b_s123_1"}, csv_overrides={123: "stats_subwp_e2_0b_s123_stage6.csv"},
          ckpt_origen_overrides={123: "pruebas/subwp_s123_wp75_r2_stage4_final.zip (via e2_0b, ver nota)"},
          notas="ASIMETRIA ENTRE SEMILLAS CONFIRMADA: para s42/s524, E2.0 parte de Fase 5 r2 (verificado). "
                "Para s123, el script original 'subwp_e2_0_s123.py' (que si parte de Fase 5 r2) fracaso por olvido "
                "catastrofico del approach durante el entrenamiento de Fase 5 (return-only): produjo timeout permanente "
                "(ep_len=6000) desde el primer batch. Se sustituyo por 'subwp_e2_0b_s123.py', que carga en su lugar el "
                "checkpoint de Fase 4 r2 (no Fase 5 r2) y sobrescribe el mismo archivo de salida 'subwp_e2_0_s123_final.zip'. "
                "La cadena real para s123 es, por tanto, Fase4r2 -> E2.0(b) -> E2.1, distinta de Fase5r2 -> E2.0 -> E2.1 en s42/s524. "
                "El intento fallido original se conserva en TensorBoard/CSV bajo 'subwp_e2_0_s123_*' y se muestra en este "
                "cuaderno como caso de estudio, pero no forma parte de la cadena valida."),
    phase("SUB-WP", "E2.1", 7, "Comportamiento con peaton mediante observaciones LIDAR realistas",
          "subwp_e2_1_s{seed}_1", "stats_subwp_e2_1_s{seed}_stage6.csv",
          "pruebas/subwp_e2_0_s{seed}_final.zip", "pruebas/subwp_e2_1_s{seed}_final.zip", 6_000_000,
          tb_overrides={123: "subwp_e2_1_s123_2"},
          notas="Observacion de 40 dimensiones (36 LIDAR + 4 estado; el comentario del constructor de WebotsEnv que "
                "sugiere +8 dims de peaton es codigo muerto/desactualizado, no se aplica realmente), LIDAR a 5.0 m, "
                "sin posicion/velocidad de peaton en la observacion de la politica, objetivos dificiles ponderados 3x. "
                "MATIZ METODOLOGICO: la recompensa usa la posicion exacta del peaton via supervisor para penalizacion de "
                "proximidad y bonus de espera en el cono de salida, aunque la politica nunca la observa directamente. "
                "Para s123 existen dos carpetas TensorBoard (_1 y _2); se usa _2 por ser la que registra el rango de pasos "
                "completo hasta 6M (verificado programaticamente mas abajo, no solo por tamaño de archivo)."),
    phase("SUB-WP", "E2.2", 8, "Replanning dinamico basado en LIDAR (umbral historico 1.5-4.5 m)",
          "subwp_e2_2_s{seed}_1", "stats_subwp_e2_2_s{seed}_stage6.csv",
          "pruebas/subwp_e2_1_s{seed}_final.zip", "pruebas/subwp_e2_2_s{seed}_final.zip", 6_000_000,
          notas="Parte de E2.1. Filtro LIDAR dinamico historico en [1.5, 4.5] m (la constante compartida en webots_env.py "
                "fue modificada in-place tras E2.2b y hoy vale 2.5 m, pero no afecta a los datos ya generados de E2.2). "
                "No se presupone mejora sobre E2.1."),
    phase("SUB-WP", "E2.2b", 9, "Replanning dinamico basado en LIDAR (umbral corregido 2.5-4.5 m)",
          "subwp_e2_2b_s{seed}_1", "stats_subwp_e2_2b_s{seed}_stage6.csv",
          "pruebas/subwp_e2_1_s{seed}_final.zip", "pruebas/subwp_e2_2b_s{seed}_final.zip", 6_000_000,
          notas="Parte DIRECTAMENTE de E2.1 (no de E2.2), confirmado en el docstring del script ('Base: E2.1, no E2.2 "
                "que fallo'). Umbral minimo elevado de 1.5 a 2.5 m. No se presupone superioridad sobre E2.1."),
]

ALL_PHASES = STHWP_PHASES + SUBWP_PHASES
PHASES_BY_KEY = {(p["arch"], p["name"]): p for p in ALL_PHASES}
print(f"Fases registradas: {len(ALL_PHASES)} ({len(STHWP_PHASES)} STH-WP + {len(SUBWP_PHASES)} SUB-WP)")

Fases registradas: 18 (9 STH-WP + 9 SUB-WP)


In [4]:
# --- Resolucion de rutas por fase y semilla, y construccion del manifiesto ---

def base_dir(arch):
    return STH_BASE if arch == "STH-WP" else SUB_BASE

def tb_name_for(ph, seed):
    if seed in ph["tb_overrides"]:
        return ph["tb_overrides"][seed]
    return ph["tb_template"].format(seed=seed) if ph["tb_template"] else None

def csv_name_for(ph, seed):
    if seed in ph["csv_overrides"]:
        return ph["csv_overrides"][seed]
    return ph["csv_template"].format(seed=seed) if ph["csv_template"] else None

def ckpt_origen_for(ph, seed):
    if seed in ph["ckpt_origen_overrides"]:
        return ph["ckpt_origen_overrides"][seed]
    if ph["ckpt_origen"] == "desde cero":
        return "desde cero"
    return ph["ckpt_origen"].format(seed=seed)

def ckpt_generado_for(ph, seed):
    return ph["ckpt_generado"].format(seed=seed)

FUNCTIONAL_TAGS = ["stats/tasa_exito_%", "stats/tasa_colision_%", "stats/tasa_truncado_%",
                   "stats/tasa_estanteria_%", "rollout/ep_rew_mean", "rollout/ep_len_mean"]
PPO_TAGS = ["train/value_loss", "train/policy_gradient_loss", "train/entropy_loss",
            "train/approx_kl", "train/clip_fraction", "train/explained_variance",
            "train/learning_rate"]

manifest_rows = []
phase_data_cache = {}  # (arch, phase_name, seed) -> {"tb_dir":..., "tags":set(), "csv_df":...}

for ph in ALL_PHASES:
    base = base_dir(ph["arch"])
    for seed in ph["seeds"]:
        tb_name = tb_name_for(ph, seed)
        tb_dir, ev_files = (None, [])
        if tb_name:
            tb_dir, ev_files = find_tb_dir(base, tb_name)
        tags = list_scalar_tags(tb_dir) if tb_dir else []
        csv_name = csv_name_for(ph, seed)
        csv_path = os.path.join(base, csv_name) if csv_name else None
        csv_df = read_stats_csv(csv_path)
        max_step = max_step_raw(tb_dir) if tb_dir else None

        estado = []
        if tb_dir is None:
            estado.append("sin TensorBoard")
        elif not tags:
            estado.append("TensorBoard vacio")
        if csv_df is None:
            estado.append("sin CSV")
        estado_final = "; ".join(estado) if estado else "completo"

        metricas_disponibles = []
        for t in FUNCTIONAL_TAGS + PPO_TAGS:
            metricas_disponibles.append(t if t in tags else f"{t} (no registrada)")

        manifest_rows.append(dict(
            arquitectura=ph["arch"], fase=ph["name"], semilla=seed,
            objetivo_fase=ph["objetivo"],
            checkpoint_origen=ckpt_origen_for(ph, seed),
            checkpoint_generado=ckpt_generado_for(ph, seed),
            timesteps_previstos=ph["timesteps_previstos"],
            timesteps_registrados=max_step if max_step is not None else "no disponible",
            directorio_tensorboard=tb_dir if tb_dir else "no localizado",
            n_archivos_eventos=len(ev_files),
            csv_asociado=csv_path if (csv_path and csv_df is not None) else "no disponible",
            metricas_disponibles="; ".join(metricas_disponibles),
            estado_datos=estado_final,
            observaciones=ph["notas"],
        ))

        phase_data_cache[(ph["arch"], ph["name"], seed)] = dict(tb_dir=tb_dir, tags=set(tags), csv_df=csv_df)

training_manifest = pd.DataFrame(manifest_rows)
manifest_path = os.path.join(TAB_DIR, "training_manifest.csv")
training_manifest.to_csv(manifest_path, index=False)
print(f"Manifiesto guardado: {manifest_path} ({len(training_manifest)} filas)")
training_manifest[["arquitectura","fase","semilla","estado_datos","timesteps_registrados"]]

Manifiesto guardado: /Users/adrigarcia/tfm-robots-rl/simulacion/notebooks/resultados_entrenamiento_hasta_e2_1/training_manifest.csv (54 filas)


,arquitectura,fase,semilla,estado_datos,timesteps_registrados
0,STH-WP,Fase 1,42,completo,501760
1,STH-WP,Fase 1,123,completo,501760
2,STH-WP,Fase 1,524,completo,501760
3,STH-WP,Fase 2,42,completo,4503552
4,STH-WP,Fase 2,123,completo,4503552
5,STH-WP,Fase 2,524,completo,4503552
6,STH-WP,Fase 3v2,42,sin CSV,8505344
7,STH-WP,Fase 3v2,123,sin CSV,8505344
8,STH-WP,Fase 3v2,524,sin CSV,8505344
9,STH-WP,Fase 4v2,42,sin CSV,12507136


In [5]:
# Advertencias registradas durante la auditoria de TensorBoard
print(f"Advertencias registradas: {len(WARNINGS_LOG)}")
for w in WARNINGS_LOG:
    print(" -", w)

Advertencias registradas: 0


## 4. Cadena de entrenamiento de STH-WP

El diagrama siguiente resume la cadena verificada. Las flechas solidas indican una relacion
confirmada mediante lectura directa de `PPO.load(...)` en el script correspondiente; no se ha
dibujado ninguna relacion que no este respaldada por el codigo.

```
Fase 1 (run002, 3 semillas, desde cero, 500k)
   |
   v
Fase 2 (run002, 3 semillas, 4M)
   |
   v
Fase 3v2 (run002, s42/s123 con script verificado; s524 solo checkpoint, sin script localizado, 4M)
   |   [TensorBoard real bajo el nombre "stage3_i6_run002_s*", no "stage3v2_*"]
   v
Fase 4v2 (run002, 3 semillas, 4M)
   |
   +----------------------------------------+
   v                                          v
Fase 5 (run002, retorno puro, 2M)     Fase 6 (run003, ciclo completo, 1M)
   [rama independiente, NO             [confirmado: parte de Fase 4v2, NO de Fase 5]
    continua hacia E2.1]                     |
                                              v
                                        E2.1 (6M, LIDAR 5m, 40 dims, sin obs. peaton, hard goals 3x)
                                          |                    |
                                          v                    v
                                    E2.2 (replanning     E2.2b (replanning
                                    LIDAR 1.5-4.5m)      LIDAR 2.5-4.5m, parte
                                                          de E2.1 directo)
```

**Sobre la consecutividad de las fases 5 y 6**: la auditoria de los scripts de entrenamiento
(`train_stage5_s*.py` y `train_stage6_s*.py`) confirma que ambas fases cargan el mismo checkpoint
de origen, `run002_s{seed}_stage4v2_final`. No existe ninguna llamada `PPO.load` en el script de
Fase 6 que apunte al checkpoint de Fase 5. Por tanto, **Fase 5 y Fase 6 son ramas de entrenamiento
paralelas e independientes** con objetivos distintos (retorno puro frente a ciclo completo
end-to-end), y no una cadena secuencial. Este cuaderno no representa una relacion 4v2→5→6 que no
esta respaldada por el codigo.

In [6]:
# --- Funciones genericas de graficado por fase ---

FUNC_PLOT_SPECS = [
    ("stats/tasa_exito_%", "Tasa de exito (%)", "exito"),
    ("stats/tasa_colision_%", "Tasa de colision (%)", "colision"),
    ("stats/tasa_truncado_%", "Tasa de truncamiento (%)", "truncamiento"),
    ("rollout/ep_rew_mean", "Recompensa media de episodio", "recompensa"),
    ("rollout/ep_len_mean", "Longitud media de episodio (pasos)", "longitud"),
]
PPO_PLOT_SPECS = [
    ("train/value_loss", "Value loss", "value_loss"),
    ("train/policy_gradient_loss", "Policy gradient loss", "policy_gradient_loss"),
    ("train/entropy_loss", "Entropy loss", "entropy_loss"),
    ("train/approx_kl", "Approximate KL", "approx_kl"),
    ("train/clip_fraction", "Clip fraction", "clip_fraction"),
    ("train/explained_variance", "Explained variance", "explained_variance"),
    ("train/learning_rate", "Learning rate", "learning_rate"),
]

def slug(text):
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")

def plot_metric_grid(ph, specs, suptitle_suffix, filename_suffix):
    """Genera una figura con una subgrafica por metrica, una linea/serie por semilla,
    con valores originales atenuados, EMA(0.85) resaltada, y leyenda de semillas.
    Devuelve el DataFrame de metricas finales por semilla y metrica.
    """
    n = len(specs)
    ncols = 3
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 3.6 * nrows))
    axes = np.array(axes).reshape(-1)
    final_values = {seed: {} for seed in ph["seeds"]}
    any_data = False

    for i, (tag, label, key) in enumerate(specs):
        ax = axes[i]
        plotted = False
        for seed in ph["seeds"]:
            cache = phase_data_cache[(ph["arch"], ph["name"], seed)]
            tb_dir = cache["tb_dir"]
            df = read_scalar(tb_dir, tag) if tb_dir else pd.DataFrame()
            if df.empty:
                continue
            plotted = True
            any_data = True
            color = SEED_COLORS[seed]
            ax.plot(df["step"], df["value"], color=color, alpha=0.18, linewidth=0.8)
            smooth = ema(df["value"], alpha=0.85)
            ax.plot(df["step"], smooth, color=color, alpha=0.95, linewidth=1.6, label=f"semilla {seed}")
            final_values[seed][key] = df["value"].iloc[-5:].mean()
        if not plotted:
            ax.text(0.5, 0.5, "no registrada", ha="center", va="center", transform=ax.transAxes,
                    fontsize=10, color="gray")
        ax.set_title(label, fontsize=10)
        ax.set_xlabel("Pasos de entrenamiento (relativos a la fase)")
        ax.set_ylabel(label)
        ax.grid(True, linestyle="--", alpha=0.3)
    for j in range(n, len(axes)):
        fig.delaxes(axes[j])
    handles = [Line2D([0], [0], color=SEED_COLORS[s], lw=2, label=f"semilla {s}") for s in ph["seeds"]]
    fig.legend(handles=handles, loc="upper center", ncol=len(ph["seeds"]), bbox_to_anchor=(0.5, 1.02))
    fig.suptitle(f"{ph['arch']} — {ph['name']} — {suptitle_suffix}", y=1.06, fontsize=12)
    fig.tight_layout()
    fname = os.path.join(FIG_DIR, f"{slug(ph['arch'])}_{slug(ph['name'])}_{filename_suffix}.png")
    fig.savefig(fname, dpi=300, bbox_inches="tight")
    plt.close(fig)
    if not any_data:
        print(f"[nota] {ph['arch']} {ph['name']}: ninguna metrica de este bloque ({suptitle_suffix}) esta registrada.")
    return fname, final_values

def plot_goal_results(ph):
    """Grafica de barras con la tasa de exito final por objetivo (goal), una serie por semilla,
    a partir de los tags 'goal/goal_XX_exito_%' de TensorBoard si existen, y si no del CSV agregado.
    """
    fig, ax = plt.subplots(figsize=(11, 4))
    any_data = False
    width = 0.25
    goal_ids = [f"goal_{i:02d}" for i in range(1, 29)]
    x = np.arange(len(goal_ids))
    for k, seed in enumerate(ph["seeds"]):
        cache = phase_data_cache[(ph["arch"], ph["name"], seed)]
        vals = None
        tb_dir = cache["tb_dir"]
        if tb_dir and any(f"goal/{g}_exito_%" in cache["tags"] for g in goal_ids):
            vals = []
            for g in goal_ids:
                tag = f"goal/{g}_exito_%"
                df = read_scalar(tb_dir, tag)
                vals.append(df["value"].iloc[-1] if not df.empty else np.nan)
        elif cache["csv_df"] is not None and "tasa_exito_%" in cache["csv_df"].columns and "goal_id" in cache["csv_df"].columns:
            dfc = cache["csv_df"].set_index("goal_id")
            vals = [dfc["tasa_exito_%"].get(g, np.nan) for g in goal_ids]
        if vals is None:
            continue
        any_data = True
        ax.bar(x + (k - 1) * width, vals, width=width, color=SEED_COLORS[seed], label=f"semilla {seed}", alpha=0.85)
    if not any_data:
        plt.close(fig)
        print(f"[nota] {ph['arch']} {ph['name']}: no hay resultados por objetivo disponibles (ni TensorBoard ni CSV).")
        return None
    ax.set_xticks(x)
    ax.set_xticklabels(goal_ids, rotation=90, fontsize=7)
    ax.set_ylabel("Tasa de exito final (%)")
    ax.set_xlabel("Objetivo (estanteria)")
    ax.set_title(f"{ph['arch']} — {ph['name']} — Tasa de exito final por objetivo")
    ax.legend()
    ax.grid(True, axis="y", linestyle="--", alpha=0.3)
    fig.tight_layout()
    fname = os.path.join(FIG_DIR, f"{slug(ph['arch'])}_{slug(ph['name'])}_por_objetivo.png")
    fig.savefig(fname, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return fname

def phase_final_table(ph, final_func, final_ppo):
    rows = []
    for seed in ph["seeds"]:
        row = {"semilla": seed}
        row.update(final_func.get(seed, {}))
        row.update(final_ppo.get(seed, {}))
        rows.append(row)
    df = pd.DataFrame(rows)
    out_path = os.path.join(TAB_DIR, f"{slug(ph['arch'])}_{slug(ph['name'])}_metricas_finales.csv")
    df.to_csv(out_path, index=False)
    return df

def analyze_phase(ph, section_title=None):
    print("="*100)
    print(f"{ph['arch']} — {ph['name']}: {ph['objetivo']}")
    print("="*100)
    print(f"Checkpoint de origen: {ckpt_origen_for(ph, ph['seeds'][0]) if len(set(ckpt_origen_for(ph,s) for s in ph['seeds']))==1 else '[ver detalle por semilla en el manifiesto]'}")
    print(f"Timesteps previstos: {ph['timesteps_previstos']:,}")
    seeds_disponibles = [s for s in ph["seeds"] if phase_data_cache[(ph['arch'], ph['name'], s)]["tb_dir"] is not None]
    print(f"Semillas con TensorBoard disponible: {seeds_disponibles if seeds_disponibles else 'NINGUNA'}")
    if ph["notas"]:
        print(f"Notas de auditoria: {ph['notas']}")
    print()

    fname_func, final_func = plot_metric_grid(ph, FUNC_PLOT_SPECS, "Metricas funcionales", "funcionales")
    fname_ppo, final_ppo = plot_metric_grid(ph, PPO_PLOT_SPECS, "Metricas internas de PPO", "ppo")
    fname_goal = plot_goal_results(ph)
    tabla = phase_final_table(ph, final_func, final_ppo)
    print(f"Figura de metricas funcionales: {os.path.basename(fname_func)}")
    print(f"Figura de metricas PPO: {os.path.basename(fname_ppo)}")
    if fname_goal:
        print(f"Figura por objetivo: {os.path.basename(fname_goal)}")
    print("Tabla de metricas finales (media de los ultimos 5 puntos registrados por semilla):")
    display(tabla)
    return tabla

print("Funciones de graficado y analisis por fase definidas.")

Funciones de graficado y analisis por fase definidas.


## 5. Fases 1–6 de STH-WP

Se presentan a continuacion, para cada fase, las metricas funcionales (tasa de exito, colision,
truncamiento, recompensa, longitud de episodio) y las metricas internas de PPO, comparando las
tres semillas cuando estan disponibles. Los valores originales se muestran con opacidad reducida
y se superpone un suavizado EMA(0.85). La tabla final resume el valor medio de las ultimas
observaciones registradas de cada metrica, por semilla.

In [7]:
tabla_sth_wp_fase_1 = analyze_phase(PHASES_BY_KEY[('STH-WP', 'Fase 1')])

STH-WP — Fase 1: Aproximacion inicial a una ubicacion
Checkpoint de origen: desde cero
Timesteps previstos: 500,000
Semillas con TensorBoard disponible: [42, 123, 524]
Notas de auditoria: Existe tambien run001 (primer intento de fases 1-3), excluido de la cadena valida porque no continua a 3v2.



Figura de metricas funcionales: sth_wp_fase_1_funcionales.png
Figura de metricas PPO: sth_wp_fase_1_ppo.png
Figura por objetivo: sth_wp_fase_1_por_objetivo.png
Tabla de metricas finales (media de los ultimos 5 puntos registrados por semilla):


,semilla,exito,colision,truncamiento,recompensa,longitud,value_loss,policy_gradient_loss,entropy_loss,approx_kl,clip_fraction,explained_variance,learning_rate
0,42,94.863599,1.880936,3.255466,77.140382,248.593997,9.142612,0.001292,-1.348379,0.013073,0.123105,0.907511,0.0003
1,123,95.207060,2.321581,2.471361,77.661678,246.794000,7.608898,0.003588,-1.906708,0.024912,0.125479,0.918439,0.0003
2,524,97.947595,0.068413,1.983990,78.885181,236.072000,8.200444,0.003464,-1.983137,0.009133,0.141436,0.886891,0.0003


In [8]:
tabla_sth_wp_fase_2 = analyze_phase(PHASES_BY_KEY[('STH-WP', 'Fase 2')])

STH-WP — Fase 2: Aproximacion generalizada a las 28 ubicaciones
Checkpoint de origen: [ver detalle por semilla en el manifiesto]
Timesteps previstos: 4,000,000
Semillas con TensorBoard disponible: [42, 123, 524]



Figura de metricas funcionales: sth_wp_fase_2_funcionales.png
Figura de metricas PPO: sth_wp_fase_2_ppo.png
Figura por objetivo: sth_wp_fase_2_por_objetivo.png
Tabla de metricas finales (media de los ultimos 5 puntos registrados por semilla):


,semilla,exito,colision,truncamiento,recompensa,longitud,value_loss,policy_gradient_loss,entropy_loss,approx_kl,clip_fraction,explained_variance,learning_rate
0,42,97.710866,2.289132,0.000000,154.749020,838.578003,5.450091,0.001885,-1.518952,0.007403,0.114043,0.915573,0.0002
1,123,96.300233,3.633304,0.066463,162.849387,834.785999,5.674422,0.001927,-1.436794,0.012883,0.162187,0.829882,0.0002
2,524,99.123322,0.834934,0.041747,153.989111,826.239990,3.156410,0.001049,-1.867892,0.013206,0.172422,0.907655,0.0002


In [9]:
tabla_sth_wp_fase_3v2 = analyze_phase(PHASES_BY_KEY[('STH-WP', 'Fase 3v2')])

STH-WP — Fase 3v2: Salida desde la estanteria (version corregida v2)
Checkpoint de origen: [ver detalle por semilla en el manifiesto]
Timesteps previstos: 4,000,000
Semillas con TensorBoard disponible: [42, 123, 524]
Notas de auditoria: tb_log_name real es 'stage3_i6_...', no existe carpeta 'stage3v2_*' (trampa de nomenclatura verificada en el script). No existe train_stage3v2_s524.py en el repositorio: el checkpoint y su inferencia existen pero el script de origen para esta semilla no ha podido localizarse. No existe CSV de entrenamiento especifico de esta fase.



Figura de metricas funcionales: sth_wp_fase_3v2_funcionales.png
Figura de metricas PPO: sth_wp_fase_3v2_ppo.png
Figura por objetivo: sth_wp_fase_3v2_por_objetivo.png
Tabla de metricas finales (media de los ultimos 5 puntos registrados por semilla):


,semilla,exito,colision,truncamiento,recompensa,longitud,value_loss,policy_gradient_loss,entropy_loss,approx_kl,clip_fraction,explained_variance,learning_rate
0,42,67.281407,32.221046,0.497546,115.349803,711.729980,26.670691,-0.001930,-3.112865,0.005984,0.077842,0.826514,0.0002
1,123,74.104254,25.895745,0.000000,106.638789,717.246008,21.334690,0.000833,-3.059300,0.011721,0.107051,0.912688,0.0002
2,524,72.411757,27.588244,0.000000,83.828401,645.450012,25.445973,0.000721,-4.149858,0.010144,0.112666,0.894892,0.0002


In [10]:
tabla_sth_wp_fase_4v2 = analyze_phase(PHASES_BY_KEY[('STH-WP', 'Fase 4v2')])

STH-WP — Fase 4v2: Combinacion de aproximacion y salida
Checkpoint de origen: [ver detalle por semilla en el manifiesto]
Timesteps previstos: 4,000,000
Semillas con TensorBoard disponible: [42, 123, 524]
Notas de auditoria: No existe CSV de entrenamiento para esta fase, solo TensorBoard.



Figura de metricas funcionales: sth_wp_fase_4v2_funcionales.png
Figura de metricas PPO: sth_wp_fase_4v2_ppo.png
Figura por objetivo: sth_wp_fase_4v2_por_objetivo.png
Tabla de metricas finales (media de los ultimos 5 puntos registrados por semilla):


,semilla,exito,colision,truncamiento,recompensa,longitud,value_loss,policy_gradient_loss,entropy_loss,approx_kl,clip_fraction,explained_variance,learning_rate
0,42,82.256381,12.910347,4.833271,274.897778,1286.722021,15.231492,0.001026,-3.729064,0.008057,0.095664,0.884352,0.0001
1,123,87.816159,7.696484,4.487357,286.537006,1432.545996,15.743923,-0.000244,-3.333090,0.004151,0.045400,0.901634,0.0001
2,524,75.405144,22.275187,2.319670,278.242218,1397.373999,14.628425,0.000912,-4.163521,0.008416,0.081152,0.896105,0.0001


In [11]:
tabla_sth_wp_fase_5 = analyze_phase(PHASES_BY_KEY[('STH-WP', 'Fase 5')])

STH-WP — Fase 5: Retorno a la base (rama de retorno puro)
Checkpoint de origen: [ver detalle por semilla en el manifiesto]
Timesteps previstos: 2,000,000
Semillas con TensorBoard disponible: [42, 123, 524]
Notas de auditoria: Parte directamente de Fase 4v2. NO continua hacia Fase 6: son ramas paralelas independientes, verificado en train_stage6_s*.py.



Figura de metricas funcionales: sth_wp_fase_5_funcionales.png
Figura de metricas PPO: sth_wp_fase_5_ppo.png
Figura por objetivo: sth_wp_fase_5_por_objetivo.png
Tabla de metricas finales (media de los ultimos 5 puntos registrados por semilla):


,semilla,exito,colision,truncamiento,recompensa,longitud,value_loss,policy_gradient_loss,entropy_loss,approx_kl,clip_fraction,explained_variance,learning_rate
0,42,99.697098,0.302904,0.0,123.118291,483.687994,9.996895,0.002309,-6.693346,0.017312,0.115000,0.981423,0.0003
1,123,100.000000,0.000000,0.0,134.321729,450.946002,12.546318,0.001250,-6.116750,0.012960,0.105566,0.978708,0.0003
2,524,89.959424,10.040575,0.0,135.589090,512.937988,8.645163,0.000158,-6.438748,0.006653,0.087988,0.981593,0.0003


In [12]:
tabla_sth_wp_fase_6 = analyze_phase(PHASES_BY_KEY[('STH-WP', 'Fase 6')])

STH-WP — Fase 6: Ciclo completo (aproximacion + salida + retorno, entrenamiento end-to-end)
Checkpoint de origen: [ver detalle por semilla en el manifiesto]
Timesteps previstos: 1,000,000
Semillas con TensorBoard disponible: [42, 123, 524]
Notas de auditoria: CONFIRMADO: Fase 6 carga el checkpoint de Fase 4v2, NO el de Fase 5. Fases 5 y 6 son ramas paralelas desde un mismo origen comun, no una cadena secuencial 4v2->5->6. Existe un intento previo descartado 'run002_s{42,524}_stage6_final' (docstring desactualizado, ausente para s123) que NO se usa en este cuaderno.



Figura de metricas funcionales: sth_wp_fase_6_funcionales.png
Figura de metricas PPO: sth_wp_fase_6_ppo.png
Figura por objetivo: sth_wp_fase_6_por_objetivo.png
Tabla de metricas finales (media de los ultimos 5 puntos registrados por semilla):


,semilla,exito,colision,truncamiento,recompensa,longitud,value_loss,policy_gradient_loss,entropy_loss,approx_kl,clip_fraction,explained_variance,learning_rate
0,42,100.000000,0.000000,0.0,507.584979,2255.504004,15.523442,-0.000189,-4.684104,0.007242,0.054287,0.914193,0.0001
1,123,98.886401,1.113597,0.0,511.444238,2287.631982,11.284939,0.000387,-4.129591,0.005564,0.083525,0.891732,0.0001
2,524,92.321234,7.678765,0.0,359.190607,1922.423975,28.229785,-0.000875,-5.048715,0.005812,0.055273,0.879109,0.0001


### Interpretacion — Fases 1 a 6 de STH-WP

Cada fase incorpora una habilidad adicional sobre la anterior: la Fase 1 establece la aproximacion
a una unica ubicacion; la Fase 2 generaliza esa aproximacion a las 28 ubicaciones del almacen; la
Fase 3v2 aisla la habilidad de salida desde la estanteria; la Fase 4v2 combina aproximacion y
salida en un unico episodio; y las fases 5 y 6, como se ha comprobado en la Seccion 4, son ramas
independientes que parten ambas de Fase 4v2: la Fase 5 aisla el retorno a la base, mientras que la
Fase 6 entrena el ciclo completo de principio a fin. La progresion de la tasa de exito y la
reduccion progresiva de la tasa de colision a lo largo de estas fases, cuando los datos estan
disponibles, constituyen la evidencia funcional de que el curriculo aporta la habilidad prevista en
cada paso. Las metricas internas de PPO (value loss decreciente, entropy loss decreciente sin
colapso prematuro, clip fraction estable) se emplean como diagnostico de la salud del entrenamiento,
no como sustituto de las metricas funcionales.

## 6. E2.1 de STH-WP

E2.1 introduce el comportamiento ante peatones mediante observaciones realistas basadas en LIDAR.
Se ha verificado en `experimentos/scripts/sthwp_e2_1_s{42,123,524}.py` que:

- La observacion tiene **40 dimensiones** (36 rayos LIDAR + 4 variables de estado).
- El alcance del LIDAR se amplia a **5.0 m**.
- La politica **no recibe** la posicion ni la velocidad del peaton desde el supervisor.
- El entrenamiento dura **6 000 000** de pasos.
- Los objetivos dificiles (`goal_14` a `goal_28`) se ponderan con un factor **3x** en el muestreo
  de episodios.
- El checkpoint de origen es el de **Fase 6** (`run003_s{seed}_stage6_final`), confirmado por
  `PPO.load`.

**Matiz metodologico**: aunque la politica no observa al peaton, la funcion de recompensa emplea
internamente la posicion del peaton obtenida del supervisor para penalizar la proximidad. Esto
significa que el comportamiento de evitacion aprendido no proviene unicamente de la percepcion
LIDAR de la politica, sino tambien de una señal de recompensa que sí dispone de informacion
privilegiada durante el entrenamiento. Esta distincion es relevante al interpretar la capacidad de
generalizacion del modelo en inferencia (vease el Notebook 2).

In [13]:
tabla_sthwp_e21 = analyze_phase(PHASES_BY_KEY[('STH-WP', 'E2.1')])

STH-WP — E2.1: Comportamiento con peaton mediante observaciones LIDAR realistas
Checkpoint de origen: [ver detalle por semilla en el manifiesto]
Timesteps previstos: 6,000,000
Semillas con TensorBoard disponible: [42, 123, 524]
Notas de auditoria: Observacion de 40 dimensiones (36 LIDAR + 4 estado), LIDAR ampliado a 5.0 m, sin posicion/velocidad de peaton en la observacion de la politica, objetivos dificiles (goal_14..goal_28) ponderados 3x. MATIZ METODOLOGICO: la recompensa usa internamente la posicion del peaton via supervisor aunque la politica no la observe.



Figura de metricas funcionales: sth_wp_e2_1_funcionales.png
Figura de metricas PPO: sth_wp_e2_1_ppo.png
Figura por objetivo: sth_wp_e2_1_por_objetivo.png
Tabla de metricas finales (media de los ultimos 5 puntos registrados por semilla):


,semilla,exito,colision,truncamiento,recompensa,longitud,value_loss,policy_gradient_loss,entropy_loss,approx_kl,clip_fraction,explained_variance,learning_rate
0,42,78.937921,21.062080,0.0,420.263037,2211.498096,36.830151,-0.001174,-4.569273,0.006253,0.082910,0.874461,0.0001
1,123,78.429277,21.570723,0.0,413.651874,2138.818018,5.655847,0.001467,-4.311178,0.007865,0.113867,0.948543,0.0001
2,524,58.905939,41.094061,0.0,365.691644,2086.250049,35.197577,-0.000075,-4.960928,0.005815,0.068320,0.861765,0.0001


## 7. E2.2 y E2.2b de STH-WP

Ambas ramas parten de **E2.1** de forma independiente (no de una a la otra) e incorporan
*replanning* dinamico basado en LIDAR para reaccionar a obstaculos moviles no presentes en el mapa
estatico:

- **E2.2**: filtra los rayos LIDAR cuyo rango cae en el intervalo **[1.5 m, 4.5 m]** y cuya celda es
  libre en el mapa estatico, para decidir cuando disparar una replanificacion A*.
- **E2.2b**: modifica el umbral minimo de deteccion de **1.5 m a 2.5 m**, con el objetivo declarado
  de reducir falsos positivos causados por las paredes de pasillos estrechos (~2 m de ancho), que a
  1.5 m entraban en el rango de deteccion dinamica y bloqueaban indebidamente el replanning.

El propio codigo del equipo de desarrollo (docstring de `sthwp_e2_2b_s*.py`) califica a E2.2 como
un intento que "fallo", por lo que este cuaderno no presupone que E2.2 mejore a E2.1, ni que E2.2b
supere de forma estable a E2.1: la comparacion cuantitativa se realiza exclusivamente a partir de
las metricas de entrenamiento registradas a continuacion, y se completa con las metricas de
inferencia en el Notebook 2.

In [14]:
tabla_sthwp_e2_2 = analyze_phase(PHASES_BY_KEY[('STH-WP', 'E2.2')])

STH-WP — E2.2: Replanning dinamico basado en LIDAR (umbral 1.5-4.5 m)
Checkpoint de origen: [ver detalle por semilla en el manifiesto]
Timesteps previstos: 6,000,000
Semillas con TensorBoard disponible: [42, 123, 524]
Notas de auditoria: Parte de E2.1. Filtro de deteccion dinamica LIDAR en [1.5, 4.5] m. No se presupone mejora sobre E2.1.



Figura de metricas funcionales: sth_wp_e2_2_funcionales.png
Figura de metricas PPO: sth_wp_e2_2_ppo.png
Figura por objetivo: sth_wp_e2_2_por_objetivo.png
Tabla de metricas finales (media de los ultimos 5 puntos registrados por semilla):


,semilla,exito,colision,truncamiento,recompensa,longitud,value_loss,policy_gradient_loss,entropy_loss,approx_kl,clip_fraction,explained_variance,learning_rate
0,42,67.105701,32.894300,0.0,348.794055,2083.135986,21.013601,-0.001519,-5.277095,0.009119,0.088906,0.884613,0.0001
1,123,66.418102,33.581898,0.0,391.635730,2023.665991,6.364129,0.001632,-4.754252,0.007076,0.082002,0.949142,0.0001
2,524,53.200811,46.799189,0.0,253.793701,1993.304028,30.770051,-0.001173,-5.411813,0.005962,0.061631,0.783181,0.0001


In [15]:
tabla_sthwp_e2_2b = analyze_phase(PHASES_BY_KEY[('STH-WP', 'E2.2b')])

STH-WP — E2.2b: Replanning dinamico basado en LIDAR (umbral corregido 2.5-4.5 m)
Checkpoint de origen: [ver detalle por semilla en el manifiesto]
Timesteps previstos: 6,000,000
Semillas con TensorBoard disponible: [42, 123, 524]
Notas de auditoria: Parte DIRECTAMENTE de E2.1 (no de E2.2). Umbral minimo de deteccion elevado de 1.5 a 2.5 m para reducir falsos positivos en pasillos estrechos (~2 m de ancho). No se presupone superioridad sobre E2.1.



Figura de metricas funcionales: sth_wp_e2_2b_funcionales.png
Figura de metricas PPO: sth_wp_e2_2b_ppo.png
Figura por objetivo: sth_wp_e2_2b_por_objetivo.png
Tabla de metricas finales (media de los ultimos 5 puntos registrados por semilla):


,semilla,exito,colision,truncamiento,recompensa,longitud,value_loss,policy_gradient_loss,entropy_loss,approx_kl,clip_fraction,explained_variance,learning_rate
0,42,65.655681,34.311136,0.033183,322.373785,1928.465991,13.483867,-0.000194,-5.289913,0.004759,0.061895,0.907277,0.0001
1,123,67.252835,32.747163,0.000000,348.529803,1995.426025,43.035789,-0.000798,-5.099388,0.005592,0.058516,0.796972,0.0001
2,524,60.434496,39.565504,0.000000,308.676410,2033.829956,55.447241,-0.000245,-5.515118,0.005085,0.057480,0.756019,0.0001


## 8. Cadena de entrenamiento de SUB-WP

```
Fase 1 (desde cero, 500k)
   |
   v
Fase 2 (4M)
   |
   v
Fase 3 r2 (4M)              [fases 3/4/5 sin sufijo r2: EXCLUIDAS, bug de zona de descarga]
   |
   v
Fase 4 r2 (4M)
   |
   v
Fase 5 r2 (2M)
   |
   v
E2.0 (2M, ped_obs=False)    [s42/s524: origen Fase 5 r2 -- CONFIRMADO]
   |                        [s123: origen REAL Fase 4 r2 via E2.0b -- ver Seccion 10]
   v
E2.1 (6M, ped_obs=True, LIDAR 5m, 40 dims, sin obs. peaton en la politica)
   |                    |
   v                    v
E2.2 (replanning   E2.2b (replanning LIDAR
LIDAR 1.5-4.5m)    2.5-4.5m, parte de E2.1
                    directo, no de E2.2)
```

Esta cadena coincide con la hipotesis de partida del encargo, salvo por la asimetria de la
semilla 123 en el salto Fase 5 r2 → E2.0, documentada en detalle en la Seccion 10. La correccion
r2 en las fases 3, 4 y 5 resuelve un error de definicion de la zona de descarga (`dropoff`) que,
segun el docstring del propio script de correccion, invalidaba las fases de salida y retorno de
todos los modelos anteriores a la correccion.

## 9. Fases 1–5 r2 de SUB-WP

In [16]:
tabla_sub_wp_fase_1 = analyze_phase(PHASES_BY_KEY[('SUB-WP', 'Fase 1')])

SUB-WP — Fase 1: Aproximacion inicial a una ubicacion
Checkpoint de origen: desde cero
Timesteps previstos: 500,000
Semillas con TensorBoard disponible: [42, 123, 524]
Notas de auditoria: Para s524 existe tambien 'stage1_subwp_s524_2' (ejecucion independiente duplicada, no una continuacion).



Figura de metricas funcionales: sub_wp_fase_1_funcionales.png
Figura de metricas PPO: sub_wp_fase_1_ppo.png
Figura por objetivo: sub_wp_fase_1_por_objetivo.png
Tabla de metricas finales (media de los ultimos 5 puntos registrados por semilla):


,semilla,exito,colision,truncamiento,recompensa,longitud,value_loss,policy_gradient_loss,entropy_loss,approx_kl,clip_fraction,explained_variance,learning_rate
0,42,98.850046,0.338222,0.811732,67.219632,266.976001,24.333313,-0.000714,-2.144501,0.010056,0.096787,0.542266,0.0003
1,123,96.612737,2.032358,1.354906,64.754811,305.514001,31.492017,0.000968,-1.815404,0.006698,0.111826,0.286993,0.0003
2,524,98.061125,1.203440,0.735436,72.219667,245.324002,23.524910,-0.001752,-2.090167,0.006717,0.069766,0.336791,0.0003


In [17]:
tabla_sub_wp_fase_2 = analyze_phase(PHASES_BY_KEY[('SUB-WP', 'Fase 2')])

SUB-WP — Fase 2: Aproximacion generalizada a las 28 ubicaciones
Checkpoint de origen: [ver detalle por semilla en el manifiesto]
Timesteps previstos: 4,000,000
Semillas con TensorBoard disponible: [42, 123, 524]



Figura de metricas funcionales: sub_wp_fase_2_funcionales.png
Figura de metricas PPO: sub_wp_fase_2_ppo.png
Figura por objetivo: sub_wp_fase_2_por_objetivo.png
Tabla de metricas finales (media de los ultimos 5 puntos registrados por semilla):


,semilla,exito,colision,truncamiento,recompensa,longitud,value_loss,policy_gradient_loss,entropy_loss,approx_kl,clip_fraction,explained_variance,learning_rate
0,42,98.614372,0.446219,0.939409,155.282748,888.480005,12.365788,0.000857,-0.544237,0.010277,0.127617,0.346682,0.0002
1,123,97.641925,1.961390,0.396686,152.732489,878.756006,12.137876,-0.000130,-1.874339,0.009026,0.101738,0.281353,0.0002
2,524,98.669444,0.798333,0.532222,164.641699,876.047998,8.848198,-0.001565,-0.948629,0.008612,0.099648,0.286160,0.0002


In [18]:
tabla_sub_wp_fase_3_r2 = analyze_phase(PHASES_BY_KEY[('SUB-WP', 'Fase 3 r2')])

SUB-WP — Fase 3 r2: Salida desde la estanteria (correccion r2 de la zona de descarga)
Checkpoint de origen: [ver detalle por semilla en el manifiesto]
Timesteps previstos: 4,000,000
Semillas con TensorBoard disponible: [42, 123, 524]
Notas de auditoria: La correccion r2 resuelve un error en la zona de descarga (dropoff) que invalidaba las fases de salida y retorno anteriores (documentado explicitamente en el docstring del script). Se excluyen las versiones sin r2. Para s42 existe tambien 'stage3_r2_subwp_s42_wp75_0' (sin sufijo noped), un intento mas corto (~4.6M pasos acumulados) que no llega al mismo punto que la version 'noped' (~8.5M acumulados, coherente con las otras semillas); se usa la version 'noped' como canonica.



Figura de metricas funcionales: sub_wp_fase_3_r2_funcionales.png
Figura de metricas PPO: sub_wp_fase_3_r2_ppo.png
Figura por objetivo: sub_wp_fase_3_r2_por_objetivo.png
Tabla de metricas finales (media de los ultimos 5 puntos registrados por semilla):


,semilla,exito,colision,truncamiento,recompensa,longitud,value_loss,policy_gradient_loss,entropy_loss,approx_kl,clip_fraction,explained_variance,learning_rate
0,42,86.766132,11.008068,2.225801,249.119266,1771.619995,25.311749,0.004117,-2.200001,0.017424,0.155078,0.201146,0.0002
1,123,90.466179,8.643564,0.890257,226.624658,1431.831982,27.471465,-0.000866,-4.353905,0.006861,0.086826,0.669742,0.0002
2,524,90.802534,8.668407,0.529058,231.996613,1463.026025,16.674833,-0.000367,-3.335884,0.008711,0.071807,0.500421,0.0002


In [19]:
tabla_sub_wp_fase_4_r2 = analyze_phase(PHASES_BY_KEY[('SUB-WP', 'Fase 4 r2')])

SUB-WP — Fase 4 r2: Ciclo aproximacion + salida (70/30)
Checkpoint de origen: [ver detalle por semilla en el manifiesto]
Timesteps previstos: 4,000,000
Semillas con TensorBoard disponible: [42, 123, 524]



Figura de metricas funcionales: sub_wp_fase_4_r2_funcionales.png
Figura de metricas PPO: sub_wp_fase_4_r2_ppo.png
Figura por objetivo: sub_wp_fase_4_r2_por_objetivo.png
Tabla de metricas finales (media de los ultimos 5 puntos registrados por semilla):


,semilla,exito,colision,truncamiento,recompensa,longitud,value_loss,policy_gradient_loss,entropy_loss,approx_kl,clip_fraction,explained_variance,learning_rate
0,42,49.615576,2.716557,47.667867,313.110278,1997.563989,17.191668,0.000899,-3.137605,0.008479,0.111406,0.243153,0.0001
1,123,54.384547,2.953354,42.662099,301.809943,1911.727979,27.374755,0.000349,-4.509563,0.004600,0.074336,0.301597,0.0001
2,524,56.946564,0.865140,42.188297,320.187286,1945.457983,15.503136,0.000018,-3.819220,0.008598,0.070488,0.380950,0.0001


In [20]:
tabla_sub_wp_fase_5_r2 = analyze_phase(PHASES_BY_KEY[('SUB-WP', 'Fase 5 r2')])

SUB-WP — Fase 5 r2: Retorno a la base (correccion r2)
Checkpoint de origen: [ver detalle por semilla en el manifiesto]
Timesteps previstos: 2,000,000
Semillas con TensorBoard disponible: [42, 123, 524]



Figura de metricas funcionales: sub_wp_fase_5_r2_funcionales.png
Figura de metricas PPO: sub_wp_fase_5_r2_ppo.png
Figura por objetivo: sub_wp_fase_5_r2_por_objetivo.png
Tabla de metricas finales (media de los ultimos 5 puntos registrados por semilla):


,semilla,exito,colision,truncamiento,recompensa,longitud,value_loss,policy_gradient_loss,entropy_loss,approx_kl,clip_fraction,explained_variance,learning_rate
0,42,95.990477,1.957025,2.052497,33.197898,1646.480005,61.161173,-0.001702,-9.866249,0.003293,0.022227,0.365520,0.0003
1,123,87.609224,0.982533,11.408244,-141.010596,2486.159912,1.289433,-0.002451,-10.617630,0.005074,0.047012,0.482368,0.0003
2,524,99.886343,0.000000,0.113656,41.847672,1834.390015,3.520891,-0.003438,-9.265498,0.006600,0.055166,0.126525,0.0003


### Interpretacion — Fases 1 a 5 r2 de SUB-WP

La progresion de SUB-WP replica la logica curricular de STH-WP: aproximacion a una ubicacion,
generalizacion a las 28 ubicaciones, salida desde la estanteria, combinacion de ambas habilidades,
y finalmente retorno a la base. La diferencia clave respecto a STH-WP es la correccion **r2**
aplicada a las fases 3, 4 y 5, necesaria porque la version original definia incorrectamente la
zona de descarga del robot, lo que invalidaba cualquier fase que dependiera de la localizacion de
entrega. Todas las metricas mostradas en esta seccion corresponden exclusivamente a la version r2.

## 10. E2.0 de SUB-WP

E2.0 es la fase equivalente a la Fase 6 de STH-WP: entrena el ciclo completo sin observacion de
peaton (`ped_obs=False`). El peaton esta fisicamente presente en el mundo y es detectado por el
LIDAR como un obstaculo mas, pero no recibe tratamiento especifico en la observacion ni en la
recompensa.

**Asimetria confirmada entre semillas** (verificada mediante `PPO.load` en los tres scripts):

- **s42 y s524**: `subwp_e2_0_s{42,524}.py` carga el checkpoint de **Fase 5 r2**, como cabria
  esperar de la posicion curricular de E2.0.
- **s123**: el script original `subwp_e2_0_s123.py` tambien intenta partir de Fase 5 r2, pero
  **este entrenamiento fracaso**: el modelo de Fase 5 r2 para la semilla 123 sufrio un olvido
  catastrofico de la habilidad de aproximacion durante el entrenamiento de retorno puro, de modo
  que al reiniciar en Fase 6 (que siempre comienza con una aproximacion) el modelo agotaba el
  limite de pasos por episodio (truncamiento permanente) desde el primer lote de entrenamiento, sin
  recompensa positiva en 2 millones de pasos. La correccion, implementada en
  `subwp_e2_0b_s123.py`, carga en su lugar el checkpoint de **Fase 4 r2** (que si domina
  aproximacion y salida) y sobrescribe el mismo archivo de salida `subwp_e2_0_s123_final.zip`.

Esta asimetria implica que, a partir de E2.0 inclusive, **la semilla 123 de SUB-WP no es
comparable en igualdad de condiciones curriculares** con las semillas 42 y 524: parte de un punto
distinto de la cadena (Fase 4 r2 en lugar de Fase 5 r2). Se muestra a continuacion, en primer
lugar, el intento fallido original como caso de estudio, y despues la comparacion de la version
valida (fallido para s123 sustituido por su correccion) entre las tres semillas.

In [21]:
# Caso de estudio: intento fallido original de E2.0 para la semilla 123
ph_fallido = phase("SUB-WP", "E2.0 (intento fallido s123)", 6, "Intento original fracasado (solo diagnostico)",
                    "subwp_e2_0_s123_1", "stats_subwp_e2_0_s123_stage6.csv",
                    "pruebas/subwp_s123_wp75_r2_stage5_final.zip", "no continua (descartado)", 2_000_000,
                    seeds=[123])
base = base_dir(ph_fallido["arch"])
tb_dir, ev_files = find_tb_dir(base, tb_name_for(ph_fallido, 123))
tags = list_scalar_tags(tb_dir) if tb_dir else []
csv_df = read_stats_csv(os.path.join(base, csv_name_for(ph_fallido, 123)) if csv_name_for(ph_fallido, 123) else None)
phase_data_cache[("SUB-WP", "E2.0 (intento fallido s123)", 123)] = dict(tb_dir=tb_dir, tags=set(tags), csv_df=csv_df)

if tb_dir:
    df_rew = read_scalar(tb_dir, "rollout/ep_rew_mean")
    df_len = read_scalar(tb_dir, "rollout/ep_len_mean")
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    axes[0].plot(df_rew["step"], df_rew["value"], color=SEED_COLORS[123])
    axes[0].set_title("Recompensa media (intento fallido, s123)")
    axes[0].set_xlabel("Pasos"); axes[0].set_ylabel("Recompensa")
    axes[1].plot(df_len["step"], df_len["value"], color=SEED_COLORS[123])
    axes[1].set_title("Longitud media de episodio (intento fallido, s123)")
    axes[1].set_xlabel("Pasos"); axes[1].set_ylabel("Pasos por episodio")
    fig.suptitle("SUB-WP -- E2.0, intento original fracasado (semilla 123) -- solo diagnostico")
    fig.tight_layout()
    fname = os.path.join(FIG_DIR, "sub_wp_e2_0_intento_fallido_s123_diagnostico.png")
    fig.savefig(fname, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Figura de diagnostico guardada: {os.path.basename(fname)}")
    print(f"Recompensa media final registrada: {df_rew['value'].iloc[-5:].mean():.3f}" if not df_rew.empty else "Recompensa: no registrada")
    print(f"Longitud media final registrada: {df_len['value'].iloc[-5:].mean():.1f} pasos (limite de episodio: truncamiento si se satura)" if not df_len.empty else "Longitud: no registrada")
else:
    print("No se ha localizado el TensorBoard del intento fallido original de E2.0 (s123).")

Figura de diagnostico guardada: sub_wp_e2_0_intento_fallido_s123_diagnostico.png
Recompensa media final registrada: -155.368
Longitud media final registrada: 3971.0 pasos (limite de episodio: truncamiento si se satura)


In [22]:
tabla_subwp_e20 = analyze_phase(PHASES_BY_KEY[('SUB-WP', 'E2.0')])

SUB-WP — E2.0: Ciclo completo sin observacion de peaton (equivalente a Fase 6 de STH-WP)
Checkpoint de origen: [ver detalle por semilla en el manifiesto]
Timesteps previstos: 2,000,000
Semillas con TensorBoard disponible: [42, 123, 524]
Notas de auditoria: ASIMETRIA ENTRE SEMILLAS CONFIRMADA: para s42/s524, E2.0 parte de Fase 5 r2 (verificado). Para s123, el script original 'subwp_e2_0_s123.py' (que si parte de Fase 5 r2) fracaso por olvido catastrofico del approach durante el entrenamiento de Fase 5 (return-only): produjo timeout permanente (ep_len=6000) desde el primer batch. Se sustituyo por 'subwp_e2_0b_s123.py', que carga en su lugar el checkpoint de Fase 4 r2 (no Fase 5 r2) y sobrescribe el mismo archivo de salida 'subwp_e2_0_s123_final.zip'. La cadena real para s123 es, por tanto, Fase4r2 -> E2.0(b) -> E2.1, distinta de Fase5r2 -> E2.0 -> E2.1 en s42/s524. El intento fallido original se conserva en TensorBoard/CSV bajo 'subwp_e2_0_s123_*' y se muestra en este cuaderno como caso 

Figura de metricas funcionales: sub_wp_e2_0_funcionales.png
Figura de metricas PPO: sub_wp_e2_0_ppo.png
Figura por objetivo: sub_wp_e2_0_por_objetivo.png
Tabla de metricas finales (media de los ultimos 5 puntos registrados por semilla):


,semilla,exito,colision,truncamiento,recompensa,longitud,value_loss,policy_gradient_loss,entropy_loss,approx_kl,clip_fraction,explained_variance,learning_rate
0,42,67.323193,29.543392,3.133414,412.722382,2741.61001,56.199388,0.000285,-6.993680,0.011057,0.101953,0.009925,0.0003
1,123,80.342683,19.657317,0.000000,511.077637,2855.02002,47.475506,-0.000233,-4.591642,0.007264,0.070928,0.266893,0.0001
2,524,73.273154,22.973077,3.753771,419.092639,2701.22207,24.566826,0.001467,-6.945430,0.013550,0.122139,-0.329511,0.0003


## 11. E2.1 de SUB-WP

Verificado en `experimentos/scripts/subwp_e2_1_s{42,123,524}.py` y en `webots_env.py`:

- Observacion de **40 dimensiones** (`obs_size = LIDAR_RAYS + 4`; el comentario del constructor de
  `WebotsEnv` que sugiere 8 dimensiones adicionales de peaton es codigo muerto/documentacion
  desactualizada y no se refleja en el calculo real de la observacion).
- LIDAR con alcance de **5.0 m** (`MAX_LIDAR_RANGE = 5.0`).
- La politica no recibe posicion ni velocidad del peaton.
- **6 000 000** de pasos de entrenamiento.
- Objetivos dificiles (`goal_17` a `goal_28`, subconjunto especifico de 10 objetivos) ponderados
  3x.
- Para s42/s524 parte de **E2.0** (Fase 5 r2 → E2.0); para s123 parte de facto de **E2.0b**
  (Fase 4 r2 → E2.0b), como se ha documentado en la Seccion 10.

**Matiz metodologico**: al igual que en STH-WP, la funcion de recompensa de E2.1 en SUB-WP utiliza
la posicion exacta del peaton obtenida del supervisor (`ped_node.getField("translation")`) para
aplicar penalizaciones de proximidad y un bonus de espera en el cono de salida, aunque la politica
nunca observa esta informacion directamente. Esta es una fuente legitima de comportamiento de
evitacion aprendido que no proviene unicamente de la percepcion LIDAR de la politica.

In [23]:
tabla_subwp_e21 = analyze_phase(PHASES_BY_KEY[('SUB-WP', 'E2.1')])

SUB-WP — E2.1: Comportamiento con peaton mediante observaciones LIDAR realistas
Checkpoint de origen: [ver detalle por semilla en el manifiesto]
Timesteps previstos: 6,000,000
Semillas con TensorBoard disponible: [42, 123, 524]
Notas de auditoria: Observacion de 40 dimensiones (36 LIDAR + 4 estado; el comentario del constructor de WebotsEnv que sugiere +8 dims de peaton es codigo muerto/desactualizado, no se aplica realmente), LIDAR a 5.0 m, sin posicion/velocidad de peaton en la observacion de la politica, objetivos dificiles ponderados 3x. MATIZ METODOLOGICO: la recompensa usa la posicion exacta del peaton via supervisor para penalizacion de proximidad y bonus de espera en el cono de salida, aunque la politica nunca la observa directamente. Para s123 existen dos carpetas TensorBoard (_1 y _2); se usa _2 por ser la que registra el rango de pasos completo hasta 6M (verificado programaticamente mas abajo, no solo por tamaño de archivo).



Figura de metricas funcionales: sub_wp_e2_1_funcionales.png
Figura de metricas PPO: sub_wp_e2_1_ppo.png
Figura por objetivo: sub_wp_e2_1_por_objetivo.png
Tabla de metricas finales (media de los ultimos 5 puntos registrados por semilla):


,semilla,exito,colision,truncamiento,recompensa,longitud,value_loss,policy_gradient_loss,entropy_loss,approx_kl,clip_fraction,explained_variance,learning_rate
0,42,73.649423,26.350577,0.0,421.935529,2814.875977,52.121389,-0.001763,-6.642674,0.004128,0.045469,0.036363,0.00005
1,123,74.820819,25.179180,0.0,448.655035,2797.063965,30.505206,-0.001317,-4.933081,0.005492,0.066289,0.273862,0.00005
2,524,74.440732,25.559269,0.0,450.928265,2884.950000,27.380664,-0.001491,-6.640262,0.004228,0.033955,0.670396,0.00005


## 12. E2.2 y E2.2b de SUB-WP

Ambas ramas parten directamente de **E2.1** (confirmado por `PPO.load`; el propio docstring de
E2.2b cita literalmente "Base: subwp_e2_1_s42_final (E2.1, no E2.2 que fallo)"):

- **E2.2**: replanning LIDAR con umbral historico **[1.5 m, 4.5 m]** (la constante compartida en
  `webots_env.py` fue modificada in-place tras E2.2b y hoy vale 2.5 m, pero esto no afecta a los
  datos de entrenamiento de E2.2 ya generados, que corresponden al valor vigente en su momento).
- **E2.2b**: umbral corregido **[2.5 m, 4.5 m]**, con el mismo objetivo que en STH-WP de reducir
  falsos positivos en pasillos estrechos.

Como en STH-WP, no se presupone que E2.2 mejore a E2.1 ni que E2.2b supere de forma estable a
E2.1; la comparacion se apoya unicamente en los datos registrados.

In [24]:
tabla_subwp_e2_2 = analyze_phase(PHASES_BY_KEY[('SUB-WP', 'E2.2')])

SUB-WP — E2.2: Replanning dinamico basado en LIDAR (umbral historico 1.5-4.5 m)
Checkpoint de origen: [ver detalle por semilla en el manifiesto]
Timesteps previstos: 6,000,000
Semillas con TensorBoard disponible: [42, 123, 524]
Notas de auditoria: Parte de E2.1. Filtro LIDAR dinamico historico en [1.5, 4.5] m (la constante compartida en webots_env.py fue modificada in-place tras E2.2b y hoy vale 2.5 m, pero no afecta a los datos ya generados de E2.2). No se presupone mejora sobre E2.1.



Figura de metricas funcionales: sub_wp_e2_2_funcionales.png
Figura de metricas PPO: sub_wp_e2_2_ppo.png
Figura por objetivo: sub_wp_e2_2_por_objetivo.png
Tabla de metricas finales (media de los ultimos 5 puntos registrados por semilla):


,semilla,exito,colision,truncamiento,recompensa,longitud,value_loss,policy_gradient_loss,entropy_loss,approx_kl,clip_fraction,explained_variance,learning_rate
0,42,63.549266,35.621470,0.829265,439.102850,2968.104004,107.243877,-0.002282,-6.326641,0.007821,0.059795,0.143549,0.0001
1,123,68.989403,31.010597,0.000000,396.938922,2723.530029,56.602780,-0.002329,-5.946924,0.006579,0.065342,0.322816,0.0001
2,524,71.361385,28.638614,0.000000,425.050165,2705.938037,67.499662,-0.000873,-6.730749,0.003815,0.046562,0.370895,0.0001


In [25]:
tabla_subwp_e2_2b = analyze_phase(PHASES_BY_KEY[('SUB-WP', 'E2.2b')])

SUB-WP — E2.2b: Replanning dinamico basado en LIDAR (umbral corregido 2.5-4.5 m)
Checkpoint de origen: [ver detalle por semilla en el manifiesto]
Timesteps previstos: 6,000,000
Semillas con TensorBoard disponible: [42, 123, 524]
Notas de auditoria: Parte DIRECTAMENTE de E2.1 (no de E2.2), confirmado en el docstring del script ('Base: E2.1, no E2.2 que fallo'). Umbral minimo elevado de 1.5 a 2.5 m. No se presupone superioridad sobre E2.1.



Figura de metricas funcionales: sub_wp_e2_2b_funcionales.png
Figura de metricas PPO: sub_wp_e2_2b_ppo.png
Figura por objetivo: sub_wp_e2_2b_por_objetivo.png
Tabla de metricas finales (media de los ultimos 5 puntos registrados por semilla):


,semilla,exito,colision,truncamiento,recompensa,longitud,value_loss,policy_gradient_loss,entropy_loss,approx_kl,clip_fraction,explained_variance,learning_rate
0,42,66.669740,33.191883,0.138376,431.148169,2880.353955,73.497054,0.000144,-6.520864,0.007472,0.064395,0.065111,0.0001
1,123,69.279291,30.675034,0.045675,301.697845,2452.433936,66.543469,-0.001981,-5.989360,0.004873,0.064980,0.253235,0.0001
2,524,69.981821,30.018177,0.000000,410.472363,2750.322021,20.886643,-0.000701,-6.456797,0.006296,0.076689,0.462536,0.0001


## 13. Evolucion curricular completa

In [26]:
# Vista curricular resumida: tasa de exito y tasa de colision finales por fase, para cada arquitectura.
# Se usa un eje categorico de fases (no un eje de pasos acumulados global) porque, como se ha
# comprobado en las auditorias, distintas fases evaluan tareas de dificultad distinta y no todas
# las fases estan encadenadas de forma estrictamente secuencial (p.ej. Fase 5 y Fase 6 en STH-WP).
# No se ha inventado ni extrapolado ningun eje de pasos acumulado global.

def curricular_summary(phases_list):
    rows = []
    for ph in phases_list:
        for seed in ph["seeds"]:
            cache = phase_data_cache[(ph["arch"], ph["name"], seed)]
            tb_dir = cache["tb_dir"]
            exito = colision = truncado = None
            if tb_dir:
                for tag, key in [("stats/tasa_exito_%", "exito"), ("stats/tasa_colision_%", "colision"),
                                   ("stats/tasa_truncado_%", "truncado")]:
                    df = read_scalar(tb_dir, tag)
                    if not df.empty:
                        if key == "exito": exito = df["value"].iloc[-5:].mean()
                        if key == "colision": colision = df["value"].iloc[-5:].mean()
                        if key == "truncado": truncado = df["value"].iloc[-5:].mean()
            rows.append(dict(fase=ph["name"], orden=ph["order"], semilla=seed,
                              exito=exito, colision=colision, truncado=truncado))
    return pd.DataFrame(rows)

for arch, phases_list in [("STH-WP", STHWP_PHASES), ("SUB-WP", SUBWP_PHASES)]:
    df_curr = curricular_summary(phases_list)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    fase_order = [p["name"] for p in sorted(phases_list, key=lambda p: p["order"])]
    for seed in SEEDS:
        sub = df_curr[df_curr["semilla"] == seed].set_index("fase").reindex(fase_order)
        axes[0].plot(fase_order, sub["exito"], marker="o", color=SEED_COLORS[seed], label=f"semilla {seed}")
        axes[1].plot(fase_order, sub["colision"], marker="o", color=SEED_COLORS[seed], label=f"semilla {seed}")
    axes[0].set_title(f"{arch} -- Tasa de exito final por fase")
    axes[0].set_ylabel("Tasa de exito (%)"); axes[0].tick_params(axis="x", rotation=45)
    axes[1].set_title(f"{arch} -- Tasa de colision final por fase")
    axes[1].set_ylabel("Tasa de colision (%)"); axes[1].tick_params(axis="x", rotation=45)
    for ax in axes:
        ax.grid(True, linestyle="--", alpha=0.3); ax.legend()
    fig.suptitle(f"Evolucion curricular resumida -- {arch}", y=1.03)
    fig.tight_layout()
    fname = os.path.join(FIG_DIR, f"{slug(arch)}_evolucion_curricular_resumida.png")
    fig.savefig(fname, dpi=300, bbox_inches="tight")
    plt.close(fig)
    df_curr.to_csv(os.path.join(TAB_DIR, f"{slug(arch)}_evolucion_curricular.csv"), index=False)
    print(f"{arch}: figura {os.path.basename(fname)} y tabla guardadas.")

STH-WP: figura sth_wp_evolucion_curricular_resumida.png y tabla guardadas.


SUB-WP: figura sub_wp_evolucion_curricular_resumida.png y tabla guardadas.


### Interpretacion de la evolucion curricular

Debe advertirse que las fases comparadas en esta vista **no evaluan la misma tarea** (aproximacion
simple, salida, ciclo completo, etc.), por lo que una caida de la tasa de exito entre fases
consecutivas no implica necesariamente un retroceso: puede reflejar simplemente el aumento de
dificultad de la tarea (por ejemplo, el ciclo completo de Fase 6/E2.0 es intrinsecamente mas dificil
que la aproximacion aislada de Fase 1). La comparacion debe leerse como una descripcion de la
dificultad relativa de cada etapa curricular, no como una serie temporal continua de aprendizaje.

## 14. Comparacion entre semillas

In [27]:
# Dispersión de la tasa de exito final entre semillas, por fase y arquitectura, para cuantificar
# la variabilidad introducida por la semilla en cada etapa del curriculo.
for arch, phases_list in [("STH-WP", STHWP_PHASES), ("SUB-WP", SUBWP_PHASES)]:
    df_curr = curricular_summary(phases_list)
    resumen = df_curr.groupby("fase")["exito"].agg(["mean", "std", "count"]).reindex(
        [p["name"] for p in sorted(phases_list, key=lambda p: p["order"])])
    resumen.columns = ["exito_medio_%", "exito_std_%", "n_semillas_con_dato"]
    print(f"--- {arch}: dispersion de tasa de exito final entre semillas ---")
    display(resumen)
    resumen.to_csv(os.path.join(TAB_DIR, f"{slug(arch)}_dispersion_entre_semillas.csv"))

--- STH-WP: dispersion de tasa de exito final entre semillas ---


,exito_medio_%,exito_std_%,n_semillas_con_dato
fase,,,
Fase 1,96.006085,1.690145,3
Fase 2,97.711474,1.411544,3
Fase 3v2,71.265806,3.552846,3
Fase 4v2,81.825895,6.216696,3
Fase 5,96.552174,5.711497,3
Fase 6,97.069212,4.149397,3
E2.1,72.091046,11.421469,3
E2.2,62.241538,7.837044,3
E2.2b,64.447671,3.566077,3


--- SUB-WP: dispersion de tasa de exito final entre semillas ---


,exito_medio_%,exito_std_%,n_semillas_con_dato
fase,,,
Fase 1,97.841302,1.134738,3
Fase 2,98.308581,0.577997,3
Fase 3 r2,89.344948,2.239644,3
Fase 4 r2,53.648896,3.720448,3
Fase 5 r2,94.495348,6.273633,3
E2.0,73.646343,6.517763,3
E2.1,74.303658,0.597607,3
E2.2,67.966684,4.005217,3
E2.2b,68.643617,1.745145,3


La desviacion estandar entre semillas por fase permite identificar en que etapas del curriculo la
inicializacion aleatoria introduce mayor variabilidad en el resultado funcional. Una desviacion
elevada en una fase concreta es una senal a tener en cuenta al interpretar comparaciones puntuales
entre configuraciones (por ejemplo, E2.1 frente a E2.2) si se basan en una unica semilla.

## 15. Comparacion entre arquitecturas

In [28]:
# Comparacion directa STH-WP vs SUB-WP en las fases conceptualmente equivalentes: Fase 6 (STH-WP)
# y E2.0 (SUB-WP) son la fase "ciclo completo sin peaton"; E2.1, E2.2, E2.2b son directamente
# comparables por nombre entre ambas arquitecturas.
comparables = [("Fase 6 / E2.0 (ciclo completo sin peaton)", "Fase 6", "E2.0"),
               ("E2.1 (con peaton)", "E2.1", "E2.1"),
               ("E2.2", "E2.2", "E2.2"),
               ("E2.2b", "E2.2b", "E2.2b")]

rows = []
for label, sth_name, sub_name in comparables:
    for arch, name in [("STH-WP", sth_name), ("SUB-WP", sub_name)]:
        ph = PHASES_BY_KEY[(arch, name)]
        for seed in ph["seeds"]:
            cache = phase_data_cache[(arch, name, seed)]
            tb_dir = cache["tb_dir"]
            exito = np.nan
            if tb_dir:
                df = read_scalar(tb_dir, "stats/tasa_exito_%")
                if not df.empty:
                    exito = df["value"].iloc[-5:].mean()
            rows.append(dict(comparacion=label, arquitectura=arch, semilla=seed, tasa_exito_final=exito))
df_comp_arch = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(9, 4.5))
labels = [c[0] for c in comparables]
x = np.arange(len(labels))
width = 0.35
for i, arch in enumerate(["STH-WP", "SUB-WP"]):
    means = [df_comp_arch[(df_comp_arch.comparacion==l) & (df_comp_arch.arquitectura==arch)]["tasa_exito_final"].mean() for l in labels]
    stds = [df_comp_arch[(df_comp_arch.comparacion==l) & (df_comp_arch.arquitectura==arch)]["tasa_exito_final"].std() for l in labels]
    ax.bar(x + (i - 0.5) * width, means, width=width, yerr=stds, capsize=4, color=ARCH_COLORS[arch], label=arch, alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=20, ha="right")
ax.set_ylabel("Tasa de exito final (%) -- media +/- desviacion entre semillas")
ax.set_title("Comparacion STH-WP vs SUB-WP en fases conceptualmente equivalentes")
ax.legend(); ax.grid(True, axis="y", linestyle="--", alpha=0.3)
fig.tight_layout()
fname = os.path.join(FIG_DIR, "comparacion_arquitecturas_sthwp_subwp.png")
fig.savefig(fname, dpi=300, bbox_inches="tight")
plt.close(fig)
df_comp_arch.to_csv(os.path.join(TAB_DIR, "comparacion_arquitecturas.csv"), index=False)
print(f"Figura guardada: {os.path.basename(fname)}")
display(df_comp_arch.pivot_table(index="comparacion", columns="arquitectura", values="tasa_exito_final", aggfunc="mean").reindex(labels))

Figura guardada: comparacion_arquitecturas_sthwp_subwp.png


arquitectura,STH-WP,SUB-WP
comparacion,,
Fase 6 / E2.0 (ciclo completo sin peaton),97.069212,73.646343
E2.1 (con peaton),72.091046,74.303658
E2.2,62.241538,67.966684
E2.2b,64.447671,68.643617


Debe recordarse que STH-WP y SUB-WP difieren en la formulacion del espacio de accion y en el
mecanismo de generacion de trayectorias (vease la memoria del TFM para el detalle de diseño), por
lo que esta comparacion cuantifica el resultado final en tareas equivalentes, pero no aisla
si la diferencia observada proviene del algoritmo de aprendizaje, del diseño del entorno, o de
ambos factores conjuntamente. Se presenta como comparacion orientativa, no como prueba causal de
superioridad de una arquitectura sobre otra.

## 16. Comparacion E2.1–E2.2–E2.2b

In [29]:
for arch in ["STH-WP", "SUB-WP"]:
    rows = []
    for name in ["E2.1", "E2.2", "E2.2b"]:
        ph = PHASES_BY_KEY[(arch, name)]
        for seed in ph["seeds"]:
            cache = phase_data_cache[(arch, name, seed)]
            tb_dir = cache["tb_dir"]
            vals = {"arquitectura": arch, "configuracion": name, "semilla": seed}
            for tag, key in [("stats/tasa_exito_%","exito"), ("stats/tasa_colision_%","colision"),
                              ("stats/tasa_truncado_%","truncado"), ("rollout/ep_rew_mean","recompensa"),
                              ("rollout/ep_len_mean","longitud")]:
                df = read_scalar(tb_dir, tag) if tb_dir else pd.DataFrame()
                vals[key] = df["value"].iloc[-5:].mean() if not df.empty else np.nan
            rows.append(vals)
    df_e2 = pd.DataFrame(rows)
    df_e2.to_csv(os.path.join(TAB_DIR, f"{slug(arch)}_comparacion_e21_e22_e22b.csv"), index=False)
    print(f"--- {arch}: E2.1 vs E2.2 vs E2.2b (medias finales por semilla) ---")
    display(df_e2)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, (key, label) in zip(axes, [("exito","Tasa de exito (%)"), ("colision","Tasa de colision (%)"), ("truncado","Tasa de truncamiento (%)")]):
        piv = df_e2.pivot(index="semilla", columns="configuracion", values=key)
        piv = piv.reindex(columns=["E2.1","E2.2","E2.2b"])
        piv.plot(kind="bar", ax=ax, color=["#4C72B0","#DD8452","#55A868"])
        ax.set_title(label); ax.set_xlabel("Semilla"); ax.legend(title=None)
        ax.grid(True, axis="y", linestyle="--", alpha=0.3)
    fig.suptitle(f"{arch} -- Comparacion E2.1 / E2.2 / E2.2b por semilla", y=1.03)
    fig.tight_layout()
    fname = os.path.join(FIG_DIR, f"{slug(arch)}_comparacion_e21_e22_e22b.png")
    fig.savefig(fname, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Figura guardada: {os.path.basename(fname)}\n")

--- STH-WP: E2.1 vs E2.2 vs E2.2b (medias finales por semilla) ---


,arquitectura,configuracion,semilla,exito,colision,truncado,recompensa,longitud
0,STH-WP,E2.1,42,78.937921,21.062080,0.000000,420.263037,2211.498096
1,STH-WP,E2.1,123,78.429277,21.570723,0.000000,413.651874,2138.818018
2,STH-WP,E2.1,524,58.905939,41.094061,0.000000,365.691644,2086.250049
3,STH-WP,E2.2,42,67.105701,32.894300,0.000000,348.794055,2083.135986
4,STH-WP,E2.2,123,66.418102,33.581898,0.000000,391.635730,2023.665991
5,STH-WP,E2.2,524,53.200811,46.799189,0.000000,253.793701,1993.304028
6,STH-WP,E2.2b,42,65.655681,34.311136,0.033183,322.373785,1928.465991
7,STH-WP,E2.2b,123,67.252835,32.747163,0.000000,348.529803,1995.426025
8,STH-WP,E2.2b,524,60.434496,39.565504,0.000000,308.676410,2033.829956


Figura guardada: sth_wp_comparacion_e21_e22_e22b.png



--- SUB-WP: E2.1 vs E2.2 vs E2.2b (medias finales por semilla) ---


,arquitectura,configuracion,semilla,exito,colision,truncado,recompensa,longitud
0,SUB-WP,E2.1,42,73.649423,26.350577,0.000000,421.935529,2814.875977
1,SUB-WP,E2.1,123,74.820819,25.179180,0.000000,448.655035,2797.063965
2,SUB-WP,E2.1,524,74.440732,25.559269,0.000000,450.928265,2884.950000
3,SUB-WP,E2.2,42,63.549266,35.621470,0.829265,439.102850,2968.104004
4,SUB-WP,E2.2,123,68.989403,31.010597,0.000000,396.938922,2723.530029
5,SUB-WP,E2.2,524,71.361385,28.638614,0.000000,425.050165,2705.938037
6,SUB-WP,E2.2b,42,66.669740,33.191883,0.138376,431.148169,2880.353955
7,SUB-WP,E2.2b,123,69.279291,30.675034,0.045675,301.697845,2452.433936
8,SUB-WP,E2.2b,524,69.981821,30.018177,0.000000,410.472363,2750.322021


Figura guardada: sub_wp_comparacion_e21_e22_e22b.png



### Preguntas orientadas por los datos (a completar con el Notebook de inferencia)

A partir de las metricas de **entrenamiento** anteriores, y pendiente de contraste con las metricas
de **inferencia con pesos congelados** del Notebook 2, se observan los siguientes hechos, sin
presuponer conclusiones no respaldadas por los datos:

1. Las tres configuraciones (E2.1, E2.2, E2.2b) alcanzan un presupuesto identico de 6 millones de
   pasos, por lo que las diferencias en las metricas finales de entrenamiento no pueden atribuirse
   a un entrenamiento mas corto de una configuracion frente a otra.
2. Cualquier mejora o empeoramiento observado en las metricas de entrenamiento debe confirmarse en
   inferencia antes de extraer conclusiones sobre generalizacion, dado que las metricas de
   entrenamiento reflejan el comportamiento durante la exploracion con la politica aun en
   actualizacion, no el comportamiento de la politica congelada final.
3. La adicion de replanning dinamico (E2.2/E2.2b) incrementa la complejidad del sistema (deteccion
   LIDAR + replanificacion A*) frente a E2.1, que no la posee. Esta complejidad adicional solo esta
   justificada si se traduce en una mejora medible y estable entre semillas, cuestion que se evalua
   con datos concretos en el Notebook 2 (Seccion "Comparacion E2.1-E2.2-E2.2b").

## 17. Seleccion de E2.1

In [30]:
print("Resumen cuantitativo de entrenamiento relevante para la seleccion de E2.1 (ver tambien Notebook 2):\n")
for arch in ["STH-WP", "SUB-WP"]:
    df_e2 = pd.read_csv(os.path.join(TAB_DIR, f"{slug(arch)}_comparacion_e21_e22_e22b.csv"))
    print(f"=== {arch} ===")
    resumen = df_e2.groupby("configuracion")[["exito","colision","truncado"]].agg(["mean","std"])
    display(resumen.reindex(["E2.1","E2.2","E2.2b"]))

Resumen cuantitativo de entrenamiento relevante para la seleccion de E2.1 (ver tambien Notebook 2):

=== STH-WP ===


exito              colision             truncado          
                    mean        std       mean        std      mean       std
configuracion                                                                
E2.1           72.091046  11.421469  27.908954  11.421469  0.000000  0.000000
E2.2           62.241538   7.837044  37.758463   7.837043  0.000000  0.000000
E2.2b          64.447671   3.566077  35.541268   3.571745  0.011061  0.019158

=== SUB-WP ===


exito             colision            truncado          
                    mean       std       mean       std      mean       std
configuracion                                                              
E2.1           74.303658  0.597607  25.696342  0.597607  0.000000  0.000000
E2.2           67.966684  4.005217  31.756894  3.550745  0.276422  0.478776
E2.2b          68.643617  1.745145  31.295032  1.675231  0.061350  0.070507

La seleccion de **E2.1** como configuracion final se apoya, a partir de la evidencia de
entrenamiento reunida en este cuaderno, en los siguientes elementos (la justificacion completa,
que incorpora ademas las metricas de inferencia con pesos congelados, se desarrolla en el
Notebook 2, Seccion "Justificacion final de E2.1"):

- E2.1 completa el curriculo completo (aproximacion, salida, retorno, ciclo completo) con
  observaciones realistas basadas unicamente en LIDAR, sin informacion privilegiada en la
  observacion de la politica, lo que la convierte en la configuracion mas alineada con un despliegue
  realista.
- E2.2 y E2.2b anaden una capa de complejidad (replanning dinamico) cuyo beneficio no esta
  garantizado por el propio historial de desarrollo del proyecto (el docstring de E2.2b describe
  a E2.2 como un intento fallido).
- La decision de mantener E2.1 como modelo final se sustenta en el conjunto de evidencia funcional,
  no unicamente en la recompensa media, evitando el sesgo de optimizar una señal que puede no
  reflejar fielmente el objetivo real de la tarea (llegar a destino sin colisionar).
- Se emplea la expresion **"no seleccionado como modelo final"** para E2.2 y E2.2b en lugar de
  "abandonado", en la medida en que ambas ramas se completaron y se evaluaron: la decision refleja
  una comparacion de resultados, no un fallo de ejecucion.

## 18. Figuras recomendadas para la memoria

In [31]:
figuras_recomendadas = [
    ("Diagrama de cadena de entrenamiento (STH-WP y SUB-WP)", "Seccion 4 y 8 (diagramas en Markdown, sin PNG dedicado)"),
    ("Evolucion curricular resumida -- STH-WP", "sth_wp_evolucion_curricular_resumida.png"),
    ("Evolucion curricular resumida -- SUB-WP", "sub_wp_evolucion_curricular_resumida.png"),
    ("Comparacion entre arquitecturas en fases equivalentes", "comparacion_arquitecturas_sthwp_subwp.png"),
    ("Comparacion E2.1/E2.2/E2.2b -- STH-WP", "sth_wp_comparacion_e21_e22_e22b.png"),
    ("Comparacion E2.1/E2.2/E2.2b -- SUB-WP", "sub_wp_comparacion_e21_e22_e22b.png"),
    ("Metricas funcionales de E2.1 -- STH-WP", "sth_wp_e2_1_funcionales.png"),
    ("Metricas funcionales de E2.1 -- SUB-WP", "sub_wp_e2_1_funcionales.png"),
    ("Diagnostico del intento fallido de E2.0 (s123, SUB-WP)", "sub_wp_e2_0_intento_fallido_s123_diagnostico.png"),
]
df_figs = pd.DataFrame(figuras_recomendadas, columns=["figura", "archivo"])
df_figs.to_csv(os.path.join(TAB_DIR, "figuras_recomendadas_memoria.csv"), index=False)
display(df_figs)

,figura,archivo
0,Diagrama de cadena de entrenamiento (STH-WP y ...,"Seccion 4 y 8 (diagramas en Markdown, sin PNG ..."
1,Evolucion curricular resumida -- STH-WP,sth_wp_evolucion_curricular_resumida.png
2,Evolucion curricular resumida -- SUB-WP,sub_wp_evolucion_curricular_resumida.png
3,Comparacion entre arquitecturas en fases equiv...,comparacion_arquitecturas_sthwp_subwp.png
4,Comparacion E2.1/E2.2/E2.2b -- STH-WP,sth_wp_comparacion_e21_e22_e22b.png
5,Comparacion E2.1/E2.2/E2.2b -- SUB-WP,sub_wp_comparacion_e21_e22_e22b.png
6,Metricas funcionales de E2.1 -- STH-WP,sth_wp_e2_1_funcionales.png
7,Metricas funcionales de E2.1 -- SUB-WP,sub_wp_e2_1_funcionales.png
8,"Diagnostico del intento fallido de E2.0 (s123,...",sub_wp_e2_0_intento_fallido_s123_diagnostico.png


## 19. Limitaciones

- Los CSV de estadisticas (`stats_*.csv`) son tablas agregadas finales por objetivo, no series
  temporales; toda la evolucion durante el entrenamiento procede de TensorBoard, cuya retencion y
  formato dependen de como se configuro cada script en su momento.
- No existe CSV de entrenamiento para la Fase 4v2 de STH-WP ni un directorio TensorBoard identificado
  de forma nativa con el nombre "stage3v2" (su registro real esta bajo "stage3_i6").
- No se ha podido localizar el script de entrenamiento de Fase 3v2 para la semilla 524 de STH-WP,
  aunque el checkpoint y su inferencia si existen; se declara como procedencia no verificable al
  100% por script, aunque el checkpoint es consistente con el resto de la cadena.
- La semilla 123 de SUB-WP sigue, a partir de E2.0, un linaje curricular distinto (Fase 4 r2 en
  lugar de Fase 5 r2) al de las semillas 42 y 524, lo que limita la comparabilidad estricta entre
  semillas en E2.0, E2.1, E2.2 y E2.2b para esta arquitectura.
- Las fases 5 y 6 de STH-WP no son secuenciales sino paralelas; cualquier lectura que las presente
  como una progresion continua seria incorrecta.
- La comparacion entre arquitecturas (Seccion 15) no controla por diferencias de diseño del entorno
  o del espacio de accion entre STH-WP y SUB-WP, por lo que su valor es orientativo.
- Este cuaderno analiza unicamente metricas de **entrenamiento**; la coherencia con el
  comportamiento en **inferencia** con pesos congelados se trata en el Notebook 2.

## 20. Conclusiones

La cadena de entrenamiento verificada confirma que tanto STH-WP como SUB-WP siguen una progresion
curricular coherente desde la aproximacion a una unica ubicacion hasta el ciclo completo con
comportamiento ante peatones (E2.1), aunque con matices importantes que solo emergen de la
inspeccion directa de los scripts: la falta de secuencialidad entre las fases 5 y 6 de STH-WP, la
trampa de nomenclatura en el registro de TensorBoard de la Fase 3v2, y la asimetria de la semilla
123 en el linaje de E2.0 de SUB-WP. Las dos ramas experimentales posteriores a E2.1, E2.2 y E2.2b,
se han entrenado con el mismo presupuesto de pasos y comparten el mismo punto de partida (E2.1),
lo que permite una comparacion directa de sus metricas de entrenamiento, presentada en la Seccion
16, sin que ello prejuzgue su comportamiento en inferencia, que se aborda en el Notebook 2. La
justificacion completa de la seleccion de E2.1 como configuracion final requiere, por tanto, la
lectura conjunta de este cuaderno y del Notebook 2 de inferencias progresivas.